# Housing Need, Demand, and Affordability

This notebook downloads, cleans, validates, and exports housing affordability
data for the 18 incorporated jurisdictions in San Diego County and for San
Diego County as a whole.

ACS refers to the American Community Survey, which provides local population and housing estimates. 

HUD refers to the U.S. Department of Housing and Urban Development, which publishes regional income limits used in housing programs.

Initial metrics:

- Median household income
- Jurisdiction-to-county income ratio
- Renter cost burden above 30%
- Severe renter cost burden above 50%
- Homeowner cost burden above 30%
- Severe homeowner cost burden above 50%
- One-bedroom median gross rent
- One-bedroom rent-to-income benchmark
- Selected monthly owner costs
- FY 2026 HUD four-person income limits

In [83]:
%pip install pandas numpy requests openpyxl

Note: you may need to restart the kernel to use updated packages.


In [84]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import requests
from IPython.display import display


def find_workstream_root() -> Path:
    """
    Find the folder containing both data/ and notebooks/.
    Works whether the notebook starts from:
    - the repository root,
    - the 'lauren's work' folder, or
    - the notebooks folder.
    """
    cwd = Path.cwd().resolve()

    candidates = [cwd, *cwd.parents]

    for candidate in candidates:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

        lauren_folder = candidate / "lauren's work"

        if (
            (lauren_folder / "data").exists()
            and (lauren_folder / "notebooks").exists()
        ):
            return lauren_folder

    raise FileNotFoundError(
        "Could not locate the workstream folder containing data/ and notebooks/."
    )


ROOT = find_workstream_root()

RAW_ACS_DIR = ROOT / "data" / "raw" / "acs"
RAW_HUD_DIR = ROOT / "data" / "raw" / "hud"
RAW_NHPD_DIR = ROOT / "data" / "raw" / "nhpd"
PROCESSED_DIR = ROOT / "data" / "processed"
DOCS_DIR = ROOT / "docs"

for folder in [
    RAW_ACS_DIR,
    RAW_HUD_DIR,
    RAW_NHPD_DIR,
    PROCESSED_DIR,
    DOCS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Workstream root:", ROOT)
print("Raw ACS folder:", RAW_ACS_DIR)
print("Raw HUD folder:", RAW_HUD_DIR)
print("Processed folder:", PROCESSED_DIR)

Workstream root: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work
Raw ACS folder: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work/data/raw/acs
Raw HUD folder: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work/data/raw/hud
Processed folder: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work/data/processed


### Set the study area and data years

Uses 2024 American Community Survey data and 2026 HUD income limits. It also defines the 18 incorporated cities in San Diego County. The Census API key allows the notebook to download data directly from the Census Bureau.

In [85]:
ACS_YEAR = 2024
HUD_YEAR = 2026

CALIFORNIA_STATE_FIPS = "06"
SAN_DIEGO_COUNTY_FIPS = "073"

from getpass import getpass

# The key is entered privately and will not be saved in the notebook.
CENSUS_API_KEY = getpass(
    "Paste your activated Census API key: "
).strip()

if not CENSUS_API_KEY:
    raise ValueError("A Census API key is required.")

ACS_GROUPS = {
    "B19013": "Median household income",
    "B25070": "Gross rent as a percentage of household income",
    "B25091": "Owner costs as a percentage of household income",
    "B25031": "Median gross rent by number of bedrooms",
    "B25088": "Median selected monthly owner costs",
}

SAN_DIEGO_CITIES = [
    "Carlsbad",
    "Chula Vista",
    "Coronado",
    "Del Mar",
    "El Cajon",
    "Encinitas",
    "Escondido",
    "Imperial Beach",
    "La Mesa",
    "Lemon Grove",
    "National City",
    "Oceanside",
    "Poway",
    "San Diego",
    "San Marcos",
    "Santee",
    "Solana Beach",
    "Vista",
]

INCLUDE_NHPD = False

print("ACS year:", ACS_YEAR)
print("Number of incorporated jurisdictions:", len(SAN_DIEGO_CITIES))

ACS year: 2024
Number of incorporated jurisdictions: 18


### Prepare Census data downloads

In [86]:
ACS_BASE_URL = (
    f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"
)


def get_json(
    url: str,
    params: dict | None = None,
    timeout: int = 60,
) -> dict | list:
    response = requests.get(
        url,
        params=params,
        timeout=timeout,
        headers={
            "User-Agent": (
                "CHPD Housing Dashboard Data Validation "
                "(housing research notebook)"
            )
        },
    )

    if not response.ok:
        raise RuntimeError(
            f"Request failed with status {response.status_code}\n"
            f"URL: {response.url}\n"
            f"Response: {response.text[:1000]}"
        )

    try:
        return response.json()

    except requests.exceptions.JSONDecodeError as error:
        raise RuntimeError(
            "The server did not return JSON.\n"
            f"URL: {response.url}\n"
            f"Content type: {response.headers.get('content-type')}\n"
            f"Response: {response.text[:1000]}"
        ) from error


def get_group_variables(group: str) -> tuple[list[str], dict]:
    """
    Retrieve all estimate and margin-of-error variables for one ACS table.

    For B25088, also retrieve the estimate annotation fields needed
    to identify medians that fall in an open-ended interval.
    """
    metadata_url = f"{ACS_BASE_URL}/groups/{group}.json"

    metadata_params = (
        {"key": CENSUS_API_KEY}
        if CENSUS_API_KEY
        else None
    )

    metadata = get_json(
        metadata_url,
        params=metadata_params,
    )

    variable_pattern = re.compile(
        rf"^{re.escape(group)}_\d{{3}}[EM]$"
    )

    variables = sorted(
        variable
        for variable in metadata["variables"]
        if variable_pattern.fullmatch(variable)
    )

    if group == "B25088":
        annotation_variables = [
            "B25088_002EA",
            "B25088_003EA",
        ]

        annotation_variables = [
            variable
            for variable in annotation_variables
            if variable in metadata["variables"]
        ]

        variables = sorted(
            set(
                variables
                + annotation_variables
            )
        )

    if not variables:
        raise ValueError(
            f"No variables found for ACS group {group}."
        )

    metadata_path = (
        RAW_ACS_DIR
        / f"acs_{ACS_YEAR}_{group}_metadata.json"
    )

    with metadata_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata,
            file,
            indent=2,
        )

    return variables, metadata

def download_acs_group(
    group: str,
    geography: str,
) -> pd.DataFrame:
    """
    Download an ACS table for:
    - all California places, or
    - San Diego County.
    """
    variables, _ = get_group_variables(group)

    params = {
        "get": ",".join(["NAME", *variables]),
    }

    if geography == "places":
        params["for"] = "place:*"
        params["in"] = f"state:{CALIFORNIA_STATE_FIPS}"

    elif geography == "county":
        params["for"] = f"county:{SAN_DIEGO_COUNTY_FIPS}"
        params["in"] = f"state:{CALIFORNIA_STATE_FIPS}"

    else:
        raise ValueError(
            "geography must be either 'places' or 'county'."
        )

    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY

    rows = get_json(ACS_BASE_URL, params=params)

    dataframe = pd.DataFrame(
        rows[1:],
        columns=rows[0],
    )

    output_path = (
        RAW_ACS_DIR
        / f"acs_{ACS_YEAR}_{group}_{geography}.csv"
    )

    dataframe.to_csv(output_path, index=False)

    print(
        f"Saved {group} {geography}: "
        f"{len(dataframe):,} rows"
    )

    return dataframe

### Download ACS data

In [87]:
acs_places_raw = {}
acs_county_raw = {}

for group, description in ACS_GROUPS.items():
    print(f"\nDownloading {group}: {description}")

    acs_places_raw[group] = download_acs_group(
        group=group,
        geography="places",
    )

    acs_county_raw[group] = download_acs_group(
        group=group,
        geography="county",
    )

print("\nAll ACS downloads completed.")


Saved B19013 places: 1,618 rows
Saved B19013 county: 1 rows

Saved B25070 places: 1,618 rows
Saved B25070 county: 1 rows

Saved B25091 places: 1,618 rows
Saved B25091 county: 1 rows

Saved B25031 places: 1,618 rows
Saved B25031 county: 1 rows

Saved B25088 places: 1,618 rows
Saved B25088 county: 1 rows

All ACS downloads completed.


In [88]:
for group in ACS_GROUPS:
    print(f"\n{group} — California places")
    display(acs_places_raw[group].head(3))

    print(f"{group} — San Diego County")
    display(acs_county_raw[group])


B19013 — California places


,NAME,B19013_001E,B19013_001M,state,place
0,"Acalanes Ridge CDP, California",181094,54426,06,00135
1,"Acampo CDP, California",-666666666,-222222222,06,00156
2,"Acton CDP, California",113000,20161,06,00212


B19013 — San Diego County


,NAME,B19013_001E,B19013_001M,state,county
0,"San Diego County, California",106268,949,06,073



B25070 — California places


...

,NAME,B25031_001E,B25031_001M,B25031_002E,B25031_002M,B25031_003E,B25031_003M,B25031_004E,B25031_004M,B25031_005E,B25031_005M,B25031_006E,B25031_006M,B25031_007E,B25031_007M,state,county
0,"San Diego County, California",2246,11,1761,35,1888,14,2290,15,2826,40,3451,82,3501,-333333333,06,073



B25088 — California places


,NAME,B25088_001E,B25088_001M,B25088_002E,B25088_002EA,B25088_002M,B25088_003E,B25088_003EA,B25088_003M,state,place
0,"Acalanes Ridge CDP, California",4001,-333333333,4001,"4,000+",-333333333,1229,NaN,202,06,00135
1,"Acampo CDP, California",-666666666,-222222222,-666666666,-,-222222222,-666666666,-,-222222222,06,00156
2,"Acton CDP, California",2672,353,3308,NaN,302,900,NaN,212,06,00212


B25088 — San Diego County


,NAME,B25088_001E,B25088_001M,B25088_002E,B25088_002EA,B25088_002M,B25088_003E,B25088_003EA,B25088_003M,state,county
0,"San Diego County, California",2535,19,3184,None,17,871,None,10,06,073


### Clean and combine the ACS tables

In [89]:
def clean_acs_numeric_values(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Convert ACS estimate and MOE columns to numeric values.
    Negative ACS special values are replaced with missing values.
    """
    cleaned = dataframe.copy()

    value_columns = [
        column
        for column in cleaned.columns
        if re.fullmatch(r"B\d{5}_\d{3}[EM]", column)
    ]

    for column in value_columns:
        cleaned[column] = pd.to_numeric(
            cleaned[column],
            errors="coerce",
        )

        cleaned.loc[
            cleaned[column] < 0,
            column,
        ] = np.nan

    return cleaned


def merge_acs_tables(
    tables: dict[str, pd.DataFrame],
    geography: str,
) -> pd.DataFrame:
    if geography == "places":
        key_columns = ["NAME", "state", "place"]
    elif geography == "county":
        key_columns = ["NAME", "state", "county"]
    else:
        raise ValueError("Invalid geography.")

    merged = None

    for group, dataframe in tables.items():
        cleaned = clean_acs_numeric_values(dataframe)

        value_columns = [
            column
            for column in cleaned.columns
            if column.startswith(f"{group}_")
        ]

        selected = cleaned[
            key_columns + value_columns
        ].copy()

        if merged is None:
            merged = selected
        else:
            merged = merged.merge(
                selected,
                on=key_columns,
                how="outer",
                validate="one_to_one",
            )

    return merged


places_wide = merge_acs_tables(
    acs_places_raw,
    geography="places",
)

county_wide = merge_acs_tables(
    acs_county_raw,
    geography="county",
)

print("California place records:", len(places_wide))
print("County records:", len(county_wide))

California place records: 1618
County records: 1


### Select San Diego County geographies

In [90]:
places_wide["jurisdiction"] = (
    places_wide["NAME"]
    .str.split(",")
    .str[0]
    .str.replace(r" city$", "", regex=True)
    .str.strip()
)

sd_places = places_wide[
    places_wide["jurisdiction"].isin(SAN_DIEGO_CITIES)
].copy()

sd_places["geoid"] = (
    sd_places["state"]
    + sd_places["place"]
)

sd_places["geography_type"] = "jurisdiction"

sd_county = county_wide.copy()

sd_county["jurisdiction"] = "San Diego County (Countywide)"

sd_county["geoid"] = (
    sd_county["state"]
    + sd_county["county"]
)

sd_county["geography_type"] = "county"

geographies = pd.concat(
    [sd_places, sd_county],
    ignore_index=True,
    sort=False,
)

geographies = geographies.sort_values(
    ["geography_type", "jurisdiction"]
).reset_index(drop=True)

missing_cities = sorted(
    set(SAN_DIEGO_CITIES)
    - set(sd_places["jurisdiction"])
)

assert not missing_cities, (
    f"Missing San Diego jurisdictions: {missing_cities}"
)

assert len(sd_places) == 18, (
    f"Expected 18 cities but found {len(sd_places)}."
)

display(
    geographies[
        [
            "geoid",
            "jurisdiction",
            "geography_type",
        ]
    ]
)

,geoid,jurisdiction,geography_type
0,06073,San Diego County (Countywide),county
1,0611194,Carlsbad,jurisdiction
2,0613392,Chula Vista,jurisdiction
3,0616378,Coronado,jurisdiction
4,0618506,Del Mar,jurisdiction
5,0621712,El Cajon,jurisdiction
6,0622678,Encinitas,jurisdiction
7,0622804,Escondido,jurisdiction
8,0636294,Imperial Beach,jurisdiction
9,0640004,La Mesa,jurisdiction


### Define dashboard geographies

The ACS output directly provides the 18 incorporated cities and a countywide San Diego County record. Unincorporated San Diego County is kept as a separate dashboard geography, but the current place-level ACS method does not provide all affordability measures directly for that area.

In [91]:
geography_dimension = pd.DataFrame(
    [
        {
            "jurisdiction": city,
            "geography_type": "incorporated jurisdiction",
            "acs_affordability_available": True,
        }
        for city in SAN_DIEGO_CITIES
    ]
    + [
        {
            "jurisdiction": "Unincorporated San Diego County",
            "geography_type": "unincorporated county",
            "acs_affordability_available": False,
        },
        {
            "jurisdiction": "San Diego County (Countywide)",
            "geography_type": "countywide",
            "acs_affordability_available": True,
        },
    ]
)

geography_dimension_path = (
    PROCESSED_DIR / "housing_dashboard_geography_dimension.csv"
)

geography_dimension.to_csv(
    geography_dimension_path,
    index=False,
)

display(geography_dimension)

,jurisdiction,geography_type,acs_affordability_available
0,Carlsbad,incorporated jurisdiction,True
1,Chula Vista,incorporated jurisdiction,True
2,Coronado,incorporated jurisdiction,True
3,Del Mar,incorporated jurisdiction,True
4,El Cajon,incorporated jurisdiction,True
5,Encinitas,incorporated jurisdiction,True
6,Escondido,incorporated jurisdiction,True
7,Imperial Beach,incorporated jurisdiction,True
8,La Mesa,incorporated jurisdiction,True
9,Lemon Grove,incorporated jurisdiction,True


### Create calculation helpers

Creates small reusable calculations used throughout the notebook. These functions add related categories, calculate percentages, and combine margins of error, which show the uncertainty in ACS estimates.

In [92]:
def sum_columns(
    dataframe: pd.DataFrame,
    columns: list[str],
) -> pd.Series:
    missing = [
        column
        for column in columns
        if column not in dataframe.columns
    ]

    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    return dataframe[columns].sum(
        axis=1,
        min_count=len(columns),
    )


def combine_margins_of_error(
    dataframe: pd.DataFrame,
    columns: list[str],
) -> pd.Series:
    """
    Approximate MOE for a sum using the root-sum-of-squares method.
    """
    missing = [
        column
        for column in columns
        if column not in dataframe.columns
    ]

    if missing:
        raise KeyError(f"Missing MOE columns: {missing}")

    squared = dataframe[columns].pow(2)

    return np.sqrt(
        squared.sum(
            axis=1,
            min_count=len(columns),
        )
    )


def safe_percentage(
    numerator: pd.Series,
    denominator: pd.Series,
) -> pd.Series:
    result = np.where(
        denominator > 0,
        numerator / denominator * 100,
        np.nan,
    )

    return pd.Series(
        result,
        index=numerator.index,
    ).round(2)

### Compare local and county incomes

In [93]:
affordability = geographies[
    [
        "geoid",
        "jurisdiction",
        "geography_type",
    ]
].copy()

affordability.insert(0, "acs_year", ACS_YEAR)
affordability["acs_period"] = "2020–2024 ACS 5-year"

affordability["median_household_income"] = (
    geographies["B19013_001E"]
)

affordability["median_household_income_moe"] = (
    geographies["B19013_001M"]
)

county_income = affordability.loc[
    affordability["geography_type"].eq("county"),
    "median_household_income",
].iloc[0]

affordability["county_median_household_income"] = (
    county_income
)

affordability["jurisdiction_county_income_ratio"] = (
    affordability["median_household_income"]
    / county_income
).round(3)

affordability[
    "jurisdiction_county_income_pct"
] = (
    affordability[
        "jurisdiction_county_income_ratio"
    ]
    * 100
).round(1)

display(
    affordability[
        [
            "jurisdiction",
            "median_household_income",
            "county_median_household_income",
            "jurisdiction_county_income_ratio",
        ]
    ].sort_values(
        "jurisdiction_county_income_ratio",
        ascending=False,
    )
)

,jurisdiction,median_household_income,county_median_household_income,jurisdiction_county_income_ratio
4,Del Mar,199152.0,106268.0,1.874
6,Encinitas,162229.0,106268.0,1.527
17,Solana Beach,152167.0,106268.0,1.432
13,Poway,148359.0,106268.0,1.396
1,Carlsbad,142748.0,106268.0,1.343
3,Coronado,134534.0,106268.0,1.266
16,Santee,113394.0,106268.0,1.067
15,San Marcos,109377.0,106268.0,1.029
2,Chula Vista,108032.0,106268.0,1.017
14,San Diego,108077.0,106268.0,1.017


### Calculate renter cost burden

Measures how many renter households spend at least 30% of their income on housing. Households spending at least 50% are considered severely cost burdened. Records where the burden could not be calculated are excluded from the percentage.

In [94]:
# B25070:
# 001 = total renter households
# 007–010 = spending at least 30% of income on gross rent
# 010 = spending at least 50%
# 011 = burden not computed

renter_burden_columns = [
    "B25070_007E",
    "B25070_008E",
    "B25070_009E",
    "B25070_010E",
]

renter_burden_moe_columns = [
    "B25070_007M",
    "B25070_008M",
    "B25070_009M",
    "B25070_010M",
]

affordability["renter_households_total"] = (
    geographies["B25070_001E"]
)

affordability["renter_households_not_computed"] = (
    geographies["B25070_011E"]
)

affordability["renter_households_computed"] = (
    affordability["renter_households_total"]
    - affordability["renter_households_not_computed"]
)

affordability["renter_cost_burdened_count"] = (
    sum_columns(
        geographies,
        renter_burden_columns,
    )
)

affordability["renter_cost_burdened_count_moe"] = (
    combine_margins_of_error(
        geographies,
        renter_burden_moe_columns,
    ).round(1)
)

affordability[
    "renter_severely_burdened_count"
] = geographies["B25070_010E"]

affordability[
    "renter_severely_burdened_count_moe"
] = geographies["B25070_010M"]

affordability["renter_cost_burdened_pct"] = (
    safe_percentage(
        affordability["renter_cost_burdened_count"],
        affordability["renter_households_computed"],
    )
)

affordability[
    "renter_severely_burdened_pct"
] = safe_percentage(
    affordability["renter_severely_burdened_count"],
    affordability["renter_households_computed"],
)

display(
    affordability[
        [
            "jurisdiction",
            "renter_households_computed",
            "renter_cost_burdened_count",
            "renter_cost_burdened_pct",
            "renter_severely_burdened_count",
            "renter_severely_burdened_pct",
        ]
    ]
)

,jurisdiction,renter_households_computed,renter_cost_burdened_count,renter_cost_burdened_pct,renter_severely_burdened_count,renter_severely_burdened_pct
0,San Diego County (Countywide),509117.0,294194.0,57.79,150516.0,29.56
1,Carlsbad,15743.0,8903.0,56.55,4471.0,28.40
2,Chula Vista,32768.0,20390.0,62.23,10970.0,33.48
3,Coronado,3248.0,2003.0,61.67,1052.0,32.39
4,Del Mar,838.0,381.0,45.47,234.0,27.92
5,El Cajon,18770.0,13157.0,70.10,7642.0,40.71
6,Encinitas,7777.0,3970.0,51.05,1957.0,25.16
7,Escondido,22276.0,13845.0,62.15,7484.0,33.60
8,Imperial Beach,6112.0,3670.0,60.05,1654.0,27.06
9,La Mesa,12590.0,7653.0,60.79,3443.0,27.35


### Calculate homeowner cost burden

Measures how many homeowner households spend at least 30% of their income on ownership costs. Households spending at least 50% are considered severely cost burdened. The calculation includes homeowners with and without a mortgage.

In [95]:
# B25091 mortgage categories:
# 002 = total with mortgage
# 008–011 = at least 30%
# 011 = at least 50%
# 012 = not computed
#
# B25091 without-mortgage categories:
# 013 = total without mortgage
# 019–022 = at least 30%
# 022 = at least 50%
# 023 = not computed

owner_mortgage_burden_columns = [
    "B25091_008E",
    "B25091_009E",
    "B25091_010E",
    "B25091_011E",
]

owner_no_mortgage_burden_columns = [
    "B25091_019E",
    "B25091_020E",
    "B25091_021E",
    "B25091_022E",
]

owner_all_burden_columns = (
    owner_mortgage_burden_columns
    + owner_no_mortgage_burden_columns
)

owner_all_burden_moe_columns = [
    column.replace("E", "M")
    for column in owner_all_burden_columns
]

affordability["owner_with_mortgage_computed"] = (
    geographies["B25091_002E"]
    - geographies["B25091_012E"]
)

affordability["owner_without_mortgage_computed"] = (
    geographies["B25091_013E"]
    - geographies["B25091_023E"]
)

affordability["owner_households_computed"] = (
    affordability["owner_with_mortgage_computed"]
    + affordability["owner_without_mortgage_computed"]
)

affordability[
    "owner_with_mortgage_burdened_count"
] = sum_columns(
    geographies,
    owner_mortgage_burden_columns,
)

affordability[
    "owner_without_mortgage_burdened_count"
] = sum_columns(
    geographies,
    owner_no_mortgage_burden_columns,
)

affordability["owner_cost_burdened_count"] = (
    sum_columns(
        geographies,
        owner_all_burden_columns,
    )
)

affordability["owner_cost_burdened_count_moe"] = (
    combine_margins_of_error(
        geographies,
        owner_all_burden_moe_columns,
    ).round(1)
)

affordability[
    "owner_with_mortgage_severely_burdened_count"
] = geographies["B25091_011E"]

affordability[
    "owner_without_mortgage_severely_burdened_count"
] = geographies["B25091_022E"]

affordability[
    "owner_severely_burdened_count"
] = (
    affordability[
        "owner_with_mortgage_severely_burdened_count"
    ]
    + affordability[
        "owner_without_mortgage_severely_burdened_count"
    ]
)

affordability[
    "owner_severely_burdened_count_moe"
] = combine_margins_of_error(
    geographies,
    [
        "B25091_011M",
        "B25091_022M",
    ],
).round(1)

affordability["owner_cost_burdened_pct"] = (
    safe_percentage(
        affordability["owner_cost_burdened_count"],
        affordability["owner_households_computed"],
    )
)

affordability[
    "owner_severely_burdened_pct"
] = safe_percentage(
    affordability["owner_severely_burdened_count"],
    affordability["owner_households_computed"],
)

affordability[
    "owner_with_mortgage_burdened_pct"
] = safe_percentage(
    affordability[
        "owner_with_mortgage_burdened_count"
    ],
    affordability["owner_with_mortgage_computed"],
)

affordability[
    "owner_without_mortgage_burdened_pct"
] = safe_percentage(
    affordability[
        "owner_without_mortgage_burdened_count"
    ],
    affordability["owner_without_mortgage_computed"],
)

affordability[
    "owner_with_mortgage_severely_burdened_pct"
] = safe_percentage(
    affordability[
        "owner_with_mortgage_severely_burdened_count"
    ],
    affordability["owner_with_mortgage_computed"],
)

affordability[
    "owner_without_mortgage_severely_burdened_pct"
] = safe_percentage(
    affordability[
        "owner_without_mortgage_severely_burdened_count"
    ],
    affordability["owner_without_mortgage_computed"],
)

display(
    affordability[
        [
            "jurisdiction",
            "owner_households_computed",
            "owner_cost_burdened_count",
            "owner_cost_burdened_pct",
            "owner_severely_burdened_count",
            "owner_severely_burdened_pct",
            "owner_with_mortgage_burdened_pct",
            "owner_with_mortgage_severely_burdened_pct",
            "owner_without_mortgage_burdened_pct",
            "owner_without_mortgage_severely_burdened_pct",
        ]
    ]
)

,jurisdiction,owner_households_computed,owner_cost_burdened_count,owner_cost_burdened_pct,owner_severely_burdened_count,owner_severely_burdened_pct,owner_with_mortgage_burdened_pct,owner_with_mortgage_severely_burdened_pct,owner_without_mortgage_burdened_pct,owner_without_mortgage_severely_burdened_pct
0,San Diego County (Countywide),633667.0,203712.0,32.15,90487.0,14.28,38.70,16.67,17.01,8.77
1,Carlsbad,27145.0,8163.0,30.07,3763.0,13.86,35.78,16.36,16.69,8.00
2,Chula Vista,50900.0,17143.0,33.68,7140.0,14.03,41.50,16.66,13.09,7.09
3,Coronado,3966.0,1533.0,38.65,860.0,21.68,49.38,30.99,25.76,10.49
4,Del Mar,987.0,259.0,26.24,157.0,15.91,40.29,24.68,8.49,4.82
5,El Cajon,14024.0,5005.0,35.69,2549.0,18.18,41.27,20.73,22.33,12.07
6,Encinitas,15820.0,4405.0,27.84,2128.0,13.45,34.46,15.36,17.10,10.35
7,Escondido,26569.0,8763.0,32.98,3621.0,13.63,38.51,14.21,20.13,12.29
8,Imperial Beach,2912.0,871.0,29.91,349.0,11.98,37.34,12.91,13.68,9.96
9,La Mesa,11876.0,3336.0,28.09,1301.0,10.95,34.61,12.72,12.86,6.82


### Measure one-bedroom rental affordability

Compares the annual cost of a typical one-bedroom rental with each jurisdiction's median household income. This is a general affordability benchmark and does not measure the actual rent burden experienced by individual households.

This is a benchmark comparison using median household income for all households. It does not represent the actual percentage of income paid by individual renter households.

In [96]:
# B25031_003 = median gross rent for a one-bedroom unit.

affordability["median_gross_rent_1br"] = (
    geographies["B25031_003E"]
)

affordability["median_gross_rent_1br_moe"] = (
    geographies["B25031_003M"]
)

affordability["annual_gross_rent_1br"] = (
    affordability["median_gross_rent_1br"] * 12
)

affordability["one_br_rent_to_income_pct"] = (
    affordability["annual_gross_rent_1br"]
    / affordability["median_household_income"]
    * 100
).round(2)

display(
    affordability[
        [
            "jurisdiction",
            "median_gross_rent_1br",
            "median_household_income",
            "one_br_rent_to_income_pct",
        ]
    ].sort_values(
        "one_br_rent_to_income_pct",
        ascending=False,
    )
)

,jurisdiction,median_gross_rent_1br,median_household_income,one_br_rent_to_income_pct
5,El Cajon,1650.0,67511.0,29.33
11,National City,1486.0,66841.0,26.68
8,Imperial Beach,1746.0,86111.0,24.33
18,Vista,1924.0,94975.0,24.31
12,Oceanside,1918.0,97737.0,23.55
9,La Mesa,1788.0,95028.0,22.58
15,San Marcos,2041.0,109377.0,22.39
7,Escondido,1683.0,91967.0,21.96
14,San Diego,1963.0,108077.0,21.80
0,San Diego County (Countywide),1888.0,106268.0,21.32


### Measure monthly homeowner costs

These ratios are benchmark comparisons using median household income for all households. They do not represent the actual housing-cost burden experienced by homeowner households.

In [97]:
# B25088_002 = median owner costs with a mortgage.
# B25088_003 = median owner costs without a mortgage.

affordability[
    "median_owner_cost_with_mortgage"
] = geographies["B25088_002E"]

affordability[
    "median_owner_cost_with_mortgage_moe"
] = geographies["B25088_002M"]

affordability[
    "median_owner_cost_without_mortgage"
] = geographies["B25088_003E"]

affordability[
    "median_owner_cost_without_mortgage_moe"
] = geographies["B25088_003M"]

affordability[
    "owner_cost_with_mortgage_to_income_pct"
] = (
    affordability["median_owner_cost_with_mortgage"]
    * 12
    / affordability["median_household_income"]
    * 100
).round(2)

affordability[
    "owner_cost_without_mortgage_to_income_pct"
] = (
    affordability["median_owner_cost_without_mortgage"]
    * 12
    / affordability["median_household_income"]
    * 100
).round(2)

affordability[
    "median_owner_cost_with_mortgage_annotation"
] = geographies["B25088_002EA"]

affordability[
    "median_owner_cost_without_mortgage_annotation"
] = geographies["B25088_003EA"]

affordability[
    "median_owner_cost_with_mortgage_top_coded"
] = affordability[
    "median_owner_cost_with_mortgage_annotation"
].eq("median+")

affordability[
    "median_owner_cost_without_mortgage_top_coded"
] = affordability[
    "median_owner_cost_without_mortgage_annotation"
].eq("median+")

affordability[
    "owner_cost_with_mortgage_to_income_is_lower_bound"
] = affordability[
    "median_owner_cost_with_mortgage_top_coded"
]

affordability[
    "owner_cost_without_mortgage_to_income_is_lower_bound"
] = affordability[
    "median_owner_cost_without_mortgage_top_coded"
]

affordability[
    "owner_cost_with_mortgage_to_income_display"
] = [
    (
        f"≥{value:.1f}%"
        if top_coded
        else f"{value:.1f}%"
    )
    if pd.notna(value)
    else None
    for value, top_coded in zip(
        affordability[
            "owner_cost_with_mortgage_to_income_pct"
        ],
        affordability[
            "median_owner_cost_with_mortgage_top_coded"
        ],
    )
]

affordability[
    "owner_cost_without_mortgage_to_income_display"
] = [
    (
        f"≥{value:.1f}%"
        if top_coded
        else f"{value:.1f}%"
    )
    if pd.notna(value)
    else None
    for value, top_coded in zip(
        affordability[
            "owner_cost_without_mortgage_to_income_pct"
        ],
        affordability[
            "median_owner_cost_without_mortgage_top_coded"
        ],
    )
]

def format_owner_cost(
    value,
    top_coded,
    top_interval_label,
):
    if pd.isna(value):
        return None

    if top_coded:
        return top_interval_label

    return f"${value:,.0f}"


affordability[
    "median_owner_cost_with_mortgage_display"
] = [
    format_owner_cost(
        value,
        top_coded,
        "$4,000+",
    )
    for value, top_coded in zip(
        affordability[
            "median_owner_cost_with_mortgage"
        ],
        affordability[
            "median_owner_cost_with_mortgage_top_coded"
        ],
    )
]

affordability[
    "median_owner_cost_without_mortgage_display"
] = [
    format_owner_cost(
        value,
        top_coded,
        "$1,500+",
    )
    for value, top_coded in zip(
        affordability[
            "median_owner_cost_without_mortgage"
        ],
        affordability[
            "median_owner_cost_without_mortgage_top_coded"
        ],
    )
]

display(
    affordability[
        [
            "jurisdiction",
            "median_owner_cost_with_mortgage",
            "median_owner_cost_without_mortgage",
            "owner_cost_with_mortgage_to_income_pct",
            "owner_cost_without_mortgage_to_income_pct",
            "median_owner_cost_with_mortgage_display",
            "median_owner_cost_without_mortgage_display",
        ]
    ]
)

,jurisdiction,median_owner_cost_with_mortgage,median_owner_cost_without_mortgage,owner_cost_with_mortgage_to_income_pct,owner_cost_without_mortgage_to_income_pct,median_owner_cost_with_mortgage_display,median_owner_cost_without_mortgage_display
0,San Diego County (Countywide),3184.0,871.0,35.95,9.84,"$3,184",$871
1,Carlsbad,3741.0,1092.0,31.45,9.18,"$3,741","$1,092"
2,Chula Vista,3090.0,806.0,34.32,8.95,"$3,090",$806
3,Coronado,4001.0,1427.0,35.69,12.73,"$4,001","$1,427"
4,Del Mar,4001.0,1208.0,24.11,7.28,"$4,001","$1,208"
5,El Cajon,2722.0,787.0,48.38,13.99,"$2,722",$787
6,Encinitas,4001.0,993.0,29.60,7.35,"$4,001",$993
7,Escondido,2869.0,850.0,37.44,11.09,"$2,869",$850
8,Imperial Beach,2659.0,597.0,37.05,8.32,"$2,659",$597
9,La Mesa,2980.0,755.0,37.63,9.53,"$2,980",$755


### Add sources and metric definitions

In [98]:
affordability["income_source_table"] = "ACS B19013"
affordability["renter_burden_source_table"] = "ACS B25070"
affordability["owner_burden_source_table"] = "ACS B25091"
affordability["one_bedroom_rent_source_table"] = "ACS B25031"
affordability["owner_cost_source_table"] = "ACS B25088"

affordability["source"] = (
    f"U.S. Census Bureau, {ACS_YEAR} ACS 5-year"
)

affordability["renter_burden_definition"] = (
    "Gross rent is at least 30% of household income; "
    "not-computed households excluded"
)

affordability["severe_renter_burden_definition"] = (
    "Gross rent is at least 50% of household income; "
    "not-computed households excluded"
)

affordability["owner_burden_definition"] = (
    "Selected monthly owner costs are at least 30% of "
    "household income; not-computed households excluded"
)

affordability["severe_owner_burden_definition"] = (
    "Selected monthly owner costs are at least 50% of "
    "household income; not-computed households excluded"
)

affordability["rent_to_income_note"] = (
    "Benchmark based on median one-bedroom gross rent divided "
    "by median household income; not an observed household burden rate"
)

affordability["rent_to_income_metric_type"] = "benchmark"

affordability["rent_to_income_denominator"] = (
    "ACS median household income for all households"
)

affordability["owner_cost_to_income_metric_type"] = (
    "benchmark"
)

affordability["owner_cost_to_income_denominator"] = (
    "ACS median household income for all households"
)

affordability["owner_cost_to_income_note"] = (
    "Benchmark comparing median selected monthly owner costs "
    "with median household income for all households; "
    "not an observed homeowner burden rate"
)

### Check the ACS results

In [99]:
assert len(
    affordability[
        affordability["geography_type"].eq("jurisdiction")
    ]
) == 18

assert affordability["geoid"].is_unique

assert affordability[
    "median_household_income"
].notna().all()

percentage_columns = [
    "renter_cost_burdened_pct",
    "renter_severely_burdened_pct",
    "owner_cost_burdened_pct",
    "owner_severely_burdened_pct",
    "owner_with_mortgage_burdened_pct",
    "owner_with_mortgage_severely_burdened_pct",
    "owner_without_mortgage_burdened_pct",
    "owner_without_mortgage_severely_burdened_pct",
]

for column in percentage_columns:
    valid_values = affordability[column].dropna()

    assert valid_values.between(0, 100).all(), (
        f"{column} contains values outside 0–100."
    )

county_ratio = affordability.loc[
    affordability["geography_type"].eq("county"),
    "jurisdiction_county_income_ratio",
].iloc[0]

assert np.isclose(county_ratio, 1.0)

print("ACS validation passed.")
print("Output rows:", len(affordability))

display(
    affordability[
        [
            "jurisdiction",
            "geography_type",
            "median_household_income",
            "renter_cost_burdened_pct",
            "renter_severely_burdened_pct",
            "owner_cost_burdened_pct",
            "owner_severely_burdened_pct",
            "median_gross_rent_1br",
            "one_br_rent_to_income_pct",
        ]
    ]
)

ACS validation passed.
Output rows: 19


,jurisdiction,geography_type,median_household_income,renter_cost_burdened_pct,renter_severely_burdened_pct,owner_cost_burdened_pct,owner_severely_burdened_pct,median_gross_rent_1br,one_br_rent_to_income_pct
0,San Diego County (Countywide),county,106268.0,57.79,29.56,32.15,14.28,1888.0,21.32
1,Carlsbad,jurisdiction,142748.0,56.55,28.40,30.07,13.86,2275.0,19.12
2,Chula Vista,jurisdiction,108032.0,62.23,33.48,33.68,14.03,1822.0,20.24
3,Coronado,jurisdiction,134534.0,61.67,32.39,38.65,21.68,2343.0,20.90
4,Del Mar,jurisdiction,199152.0,45.47,27.92,26.24,15.91,3003.0,18.09
5,El Cajon,jurisdiction,67511.0,70.10,40.71,35.69,18.18,1650.0,29.33
6,Encinitas,jurisdiction,162229.0,51.05,25.16,27.84,13.45,2233.0,16.52
7,Escondido,jurisdiction,91967.0,62.15,33.60,32.98,13.63,1683.0,21.96
8,Imperial Beach,jurisdiction,86111.0,60.05,27.06,29.91,11.98,1746.0,24.33
9,La Mesa,jurisdiction,95028.0,60.79,27.35,28.09,10.95,1788.0,22.58


### Save the processed ACS dataset

In [100]:
acs_output_path = (
    PROCESSED_DIR
    / "acs_2024_housing_need_affordability_by_jurisdiction.csv"
)

affordability.to_csv(
    acs_output_path,
    index=False,
)

print("Saved processed ACS dataset:")
print(acs_output_path)

Saved processed ACS dataset:
/Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work/data/processed/acs_2024_housing_need_affordability_by_jurisdiction.csv


### Download HUD income limits

In [101]:
HUD_EXCEL_URL = (
    "https://www.huduser.gov/portal/datasets/"
    "il/il26/Section8-FY26.xlsx"
)

HUD_EXCEL_PATH = (
    RAW_HUD_DIR
    / "Section8-FY26.xlsx"
)

response = requests.get(
    HUD_EXCEL_URL,
    timeout=120,
    headers={
        "User-Agent": (
            "Mozilla/5.0 CHPD Housing Dashboard "
            "Data Validation"
        )
    },
)

response.raise_for_status()

# XLSX files are ZIP-based and normally begin with PK.
if not response.content.startswith(b"PK"):
    raise RuntimeError(
        "HUD returned content that does not appear to be an XLSX file."
    )

HUD_EXCEL_PATH.write_bytes(response.content)

print("Downloaded HUD workbook:")
print(HUD_EXCEL_PATH)
print("File size:", f"{HUD_EXCEL_PATH.stat().st_size:,} bytes")

Downloaded HUD workbook:
/Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work/data/raw/hud/Section8-FY26.xlsx
File size: 771,324 bytes


### Read and organize the HUD workbook

In [102]:
def identify_header_row(
    raw_dataframe: pd.DataFrame,
    maximum_rows: int = 25,
) -> int:
    search_terms = [
        "state",
        "county",
        "area",
        "median",
        "fips",
        "l50",
        "income",
    ]

    best_row = 0
    best_score = -1

    for row_number in range(
        min(maximum_rows, len(raw_dataframe))
    ):
        row_text = " ".join(
            raw_dataframe.iloc[row_number]
            .fillna("")
            .astype(str)
            .str.lower()
            .tolist()
        )

        score = sum(
            term in row_text
            for term in search_terms
        )

        if score > best_score:
            best_row = row_number
            best_score = score

    return best_row


excel_file = pd.ExcelFile(
    HUD_EXCEL_PATH,
    engine="openpyxl",
)

print("HUD workbook sheets:")
print(excel_file.sheet_names)

hud_sheets = {}

for sheet_name in excel_file.sheet_names:
    raw_sheet = pd.read_excel(
        HUD_EXCEL_PATH,
        sheet_name=sheet_name,
        header=None,
        engine="openpyxl",
    )

    header_row = identify_header_row(raw_sheet)

    cleaned_sheet = pd.read_excel(
        HUD_EXCEL_PATH,
        sheet_name=sheet_name,
        header=header_row,
        engine="openpyxl",
    )

    cleaned_sheet = cleaned_sheet.dropna(
        how="all"
    ).reset_index(drop=True)

    hud_sheets[sheet_name] = cleaned_sheet

    print(
        f"\nSheet: {sheet_name} | "
        f"Header row: {header_row} | "
        f"Rows: {len(cleaned_sheet)}"
    )

    print(list(cleaned_sheet.columns))

HUD workbook sheets:
['Section8-FY26', 'Field_Descriptions']

Sheet: Section8-FY26 | Header row: 0 | Rows: 4764
['fips', 'stusps', 'state', 'state_name', 'hud_area_code', 'hud_area_name', 'county', 'County_Name', 'county_town_name', 'metro', 'median2026', 'l50_1', 'l50_2', 'l50_3', 'l50_4', 'l50_5', 'l50_6', 'l50_7', 'l50_8', 'ELI_1', 'ELI_2', 'ELI_3', 'ELI_4', 'ELI_5', 'ELI_6', 'ELI_7', 'ELI_8', 'l80_1', 'l80_2', 'l80_3', 'l80_4', 'l80_5', 'l80_6', 'l80_7', 'l80_8']

Sheet: Field_Descriptions | Header row: 10 | Rows: 25
['fips', 'Concatenated 2 Digit State FIPS Code, 3 Digit County FIPS Code, and 5 Digit County Subdivision FIPS Code']


### Find San Diego HUD records

In [103]:
hud_san_diego_matches = []

for sheet_name, dataframe in hud_sheets.items():
    text_dataframe = dataframe.fillna("").astype(str)

    contains_san_diego = text_dataframe.apply(
        lambda column: column.str.contains(
            "San Diego",
            case=False,
            regex=False,
        )
    ).any(axis=1)

    matches = dataframe[
        contains_san_diego
    ].copy()

    if not matches.empty:
        matches.insert(0, "_sheet_name", sheet_name)
        hud_san_diego_matches.append(matches)

if not hud_san_diego_matches:
    raise ValueError(
        "No San Diego rows were found in the HUD workbook."
    )

hud_san_diego = pd.concat(
    hud_san_diego_matches,
    ignore_index=True,
    sort=False,
)

display(hud_san_diego)

,_sheet_name,fips,stusps,state,state_name,hud_area_code,hud_area_name,county,County_Name,county_town_name,...,ELI_7,ELI_8,l80_1,l80_2,l80_3,l80_4,l80_5,l80_6,l80_7,l80_8
0,Section8-FY26,607399999,CA,6,California,METRO41740M41740,"San Diego-Chula Vista-Carlsbad, CA MSA",73,San Diego County,San Diego County,...,65050,69250,97950,111950,125950,139900,151100,162300,173500,184700


### Standardize HUD column names

In [104]:
def normalize_column_name(column: object) -> str:
    normalized = str(column).strip().lower()

    normalized = re.sub(
        r"[^a-z0-9]+",
        "_",
        normalized,
    )

    return normalized.strip("_")


hud_san_diego = hud_san_diego.rename(
    columns={
        column: normalize_column_name(column)
        for column in hud_san_diego.columns
    }
)

print("Normalized HUD columns:")
print(list(hud_san_diego.columns))

display(hud_san_diego)

Normalized HUD columns:
['sheet_name', 'fips', 'stusps', 'state', 'state_name', 'hud_area_code', 'hud_area_name', 'county', 'county_name', 'county_town_name', 'metro', 'median2026', 'l50_1', 'l50_2', 'l50_3', 'l50_4', 'l50_5', 'l50_6', 'l50_7', 'l50_8', 'eli_1', 'eli_2', 'eli_3', 'eli_4', 'eli_5', 'eli_6', 'eli_7', 'eli_8', 'l80_1', 'l80_2', 'l80_3', 'l80_4', 'l80_5', 'l80_6', 'l80_7', 'l80_8']


,sheet_name,fips,stusps,state,state_name,hud_area_code,hud_area_name,county,county_name,county_town_name,...,eli_7,eli_8,l80_1,l80_2,l80_3,l80_4,l80_5,l80_6,l80_7,l80_8
0,Section8-FY26,607399999,CA,6,California,METRO41740M41740,"San Diego-Chula Vista-Carlsbad, CA MSA",73,San Diego County,San Diego County,...,65050,69250,97950,111950,125950,139900,151100,162300,173500,184700


### Identify the needed HUD fields

In [105]:
def find_column(
    dataframe: pd.DataFrame,
    candidates: list[str],
    required: bool = True,
) -> str | None:
    normalized_columns = {
        normalize_column_name(column): column
        for column in dataframe.columns
    }

    # First attempt an exact match.
    for candidate in candidates:
        normalized_candidate = normalize_column_name(
            candidate
        )

        if normalized_candidate in normalized_columns:
            return normalized_columns[normalized_candidate]

    # Then attempt a partial match.
    for candidate in candidates:
        normalized_candidate = normalize_column_name(
            candidate
        )

        for normalized_column, original_column in (
            normalized_columns.items()
        ):
            if normalized_candidate in normalized_column:
                return original_column

    if required:
        raise KeyError(
            f"Could not find any of these HUD columns: {candidates}\n"
            f"Available columns: {list(dataframe.columns)}"
        )

    return None


hud_area_column = find_column(
    hud_san_diego,
    [
        "hud_area_name",
        "area_name",
        "hud_area",
    ],
    required=False,
)

hud_county_column = find_column(
    hud_san_diego,
    [
        "county_name",
        "county",
    ],
    required=False,
)

hud_median_column = find_column(
    hud_san_diego,
    [
        "median2026",
        "median_2026",
        "median_family_income",
        "median_income",
    ],
)

hud_30_four_person_column = find_column(
    hud_san_diego,
    [
        "l30_4",
        "eli_4",
        "extremely_low_income_4",
        "30_4",
    ],
)

hud_50_four_person_column = find_column(
    hud_san_diego,
    [
        "l50_4",
        "vlil_4",
        "very_low_income_4",
        "50_4",
    ],
)

hud_80_four_person_column = find_column(
    hud_san_diego,
    [
        "l80_4",
        "lil_4",
        "low_income_4",
        "80_4",
    ],
)

print("HUD area column:", hud_area_column)
print("HUD county column:", hud_county_column)
print("Median family income column:", hud_median_column)
print("Four-person 30% column:", hud_30_four_person_column)
print("Four-person 50% column:", hud_50_four_person_column)
print("Four-person 80% column:", hud_80_four_person_column)

HUD area column: hud_area_name
HUD county column: county_name
Median family income column: median2026
Four-person 30% column: eli_4
Four-person 50% column: l50_4
Four-person 80% column: l80_4


### Select the San Diego HUD record

In [106]:
hud_selected = hud_san_diego.copy()

if hud_county_column is not None:
    county_match = (
        hud_selected[hud_county_column]
        .astype(str)
        .str.contains(
            "San Diego",
            case=False,
            regex=False,
            na=False,
        )
    )

    if county_match.any():
        hud_selected = hud_selected[
            county_match
        ].copy()

# Remove exact duplicate records if the same record appeared twice.
hud_selected = hud_selected.drop_duplicates()

print("Candidate San Diego HUD records:")
display(hud_selected)

# The official file may repeat the same income-limit area by county.
# San Diego County should normally resolve to one unique limit record.
hud_record = hud_selected.iloc[0]

Candidate San Diego HUD records:


,sheet_name,fips,stusps,state,state_name,hud_area_code,hud_area_name,county,county_name,county_town_name,...,eli_7,eli_8,l80_1,l80_2,l80_3,l80_4,l80_5,l80_6,l80_7,l80_8
0,Section8-FY26,607399999,CA,6,California,METRO41740M41740,"San Diego-Chula Vista-Carlsbad, CA MSA",73,San Diego County,San Diego County,...,65050,69250,97950,111950,125950,139900,151100,162300,173500,184700


### Create the HUD output

In [107]:
def numeric_value(value: object) -> float:
    return pd.to_numeric(
        pd.Series([value]),
        errors="coerce",
    ).iloc[0]


hud_output = pd.DataFrame(
    [
        {
            "hud_fiscal_year": HUD_YEAR,
            "effective_date": "2026-05-01",
            "geography_type": "HUD income-limit area",
            "hud_area_name": (
                hud_record[hud_area_column]
                if hud_area_column is not None
                else "San Diego HUD income-limit area"
            ),
            "county_name": (
                hud_record[hud_county_column]
                if hud_county_column is not None
                else "San Diego County"
            ),
            "four_person_hud_median_family_income": (
                numeric_value(
                    hud_record[hud_median_column]
                )
            ),
            "four_person_extremely_low_income_limit": (
                numeric_value(
                    hud_record[
                        hud_30_four_person_column
                    ]
                )
            ),
            "four_person_very_low_income_limit": (
                numeric_value(
                    hud_record[
                        hud_50_four_person_column
                    ]
                )
            ),
            "four_person_low_income_limit": (
                numeric_value(
                    hud_record[
                        hud_80_four_person_column
                    ]
                )
            ),
            "source": (
                "HUD FY 2026 Section 8 Income Limits"
            ),
            "source_file": "Section8-FY26.xlsx",
            "note": (
                "HUD AMI is a regional income-limit benchmark, "
                "not a jurisdiction-specific median household income."
            ),
        }
    ]
)

hud_output_path = (
    PROCESSED_DIR
    / "hud_2026_four_person_income_limits_san_diego.csv"
)

hud_output.to_csv(
    hud_output_path,
    index=False,
)

display(hud_output)

print("Saved processed HUD dataset:")
print(hud_output_path)

,hud_fiscal_year,effective_date,geography_type,hud_area_name,county_name,four_person_hud_median_family_income,four_person_extremely_low_income_limit,four_person_very_low_income_limit,four_person_low_income_limit,source,source_file,note
0,2026,2026-05-01,HUD income-limit area,"San Diego-Chula Vista-Carlsbad, CA MSA",San Diego County,130900,52450,87450,139900,HUD FY 2026 Section 8 Income Limits,Section8-FY26.xlsx,"HUD AMI is a regional income-limit benchmark, ..."


Saved processed HUD dataset:
/Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work/data/processed/hud_2026_four_person_income_limits_san_diego.csv


### Combine ACS and HUD results

Adds the San Diego regional HUD income limits to each jurisdiction's ACS record. These HUD values are regional benchmarks, so they are the same for every city and should not be treated as city-specific median incomes.

In [108]:
combined_output = affordability.copy()

combined_output["hud_fiscal_year"] = (
    hud_output.loc[0, "hud_fiscal_year"]
)

combined_output[
    "hud_four_person_median_family_income"
] = hud_output.loc[
    0,
    "four_person_hud_median_family_income",
]

combined_output[
    "hud_four_person_extremely_low_income_limit"
] = hud_output.loc[
    0,
    "four_person_extremely_low_income_limit",
]

combined_output[
    "hud_four_person_very_low_income_limit"
] = hud_output.loc[
    0,
    "four_person_very_low_income_limit",
]

combined_output[
    "hud_four_person_low_income_limit"
] = hud_output.loc[
    0,
    "four_person_low_income_limit",
]

combined_output["hud_geography_note"] = (
    "HUD values are regional benchmarks repeated for each jurisdiction."
)

combined_output_path = (
    PROCESSED_DIR
    / "housing_need_affordability_acs2024_hud2026.csv"
)

combined_output.to_csv(
    combined_output_path,
    index=False,
)

print("Saved combined dataset:")
print(combined_output_path)

display(combined_output.head())

Saved combined dataset:
/Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work/data/processed/housing_need_affordability_acs2024_hud2026.csv


,acs_year,geoid,jurisdiction,geography_type,acs_period,median_household_income,median_household_income_moe,county_median_household_income,jurisdiction_county_income_ratio,jurisdiction_county_income_pct,...,rent_to_income_denominator,owner_cost_to_income_metric_type,owner_cost_to_income_denominator,owner_cost_to_income_note,hud_fiscal_year,hud_four_person_median_family_income,hud_four_person_extremely_low_income_limit,hud_four_person_very_low_income_limit,hud_four_person_low_income_limit,hud_geography_note
0,2024,06073,San Diego County (Countywide),county,2020–2024 ACS 5-year,106268.0,949.0,106268.0,1.000,100.0,...,ACS median household income for all households,benchmark,ACS median household income for all households,Benchmark comparing median selected monthly ow...,2026,130900,52450,87450,139900,HUD values are regional benchmarks repeated fo...
1,2024,0611194,Carlsbad,jurisdiction,2020–2024 ACS 5-year,142748.0,7547.0,106268.0,1.343,134.3,...,ACS median household income for all households,benchmark,ACS median household income for all households,Benchmark comparing median selected monthly ow...,2026,130900,52450,87450,139900,HUD values are regional benchmarks repeated fo...
2,2024,0613392,Chula Vista,jurisdiction,2020–2024 ACS 5-year,108032.0,2863.0,106268.0,1.017,101.7,...,ACS median household income for all households,benchmark,ACS median household income for all households,Benchmark comparing median selected monthly ow...,2026,130900,52450,87450,139900,HUD values are regional benchmarks repeated fo...
3,2024,0616378,Coronado,jurisdiction,2020–2024 ACS 5-year,134534.0,11789.0,106268.0,1.266,126.6,...,ACS median household income for all households,benchmark,ACS median household income for all households,Benchmark comparing median selected monthly ow...,2026,130900,52450,87450,139900,HUD values are regional benchmarks repeated fo...
4,2024,0618506,Del Mar,jurisdiction,2020–2024 ACS 5-year,199152.0,30303.0,106268.0,1.874,187.4,...,ACS median household income for all households,benchmark,ACS median household income for all households,Benchmark comparing median selected monthly ow...,2026,130900,52450,87450,139900,HUD values are regional benchmarks repeated fo...


### Federally Assisted Housing and Preservation Risk

This section uses the National Housing Preservation Database (NHPD) to identify federally assisted housing properties, subsidy programs, subsidy status, assisted-unit counts where available, and upcoming affordability expiration dates.

Property-level housing counts and subsidy-level assisted-unit counts are kept separate to avoid double counting.

In [109]:
# ============================================================
# NHPD: Federally Assisted Housing + Preservation Risk
# ============================================================

NHPD_RAW_PATH = (
    RAW_NHPD_DIR
    / "nhpd_data_extract.csv"
)

NHPD_AVAILABLE = (
    INCLUDE_NHPD
    and NHPD_RAW_PATH.exists()
)

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

nhpd_property_path = (
    PROCESSED_DIR
    / "nhpd_san_diego_properties.csv"
)

nhpd_subsidy_path = (
    PROCESSED_DIR
    / "nhpd_san_diego_subsidies.csv"
)

nhpd_summary_path = (
    PROCESSED_DIR
    / "nhpd_san_diego_by_jurisdiction.csv"
)

nhpd_subsidy_risk_path = (
    PROCESSED_DIR
    / "nhpd_san_diego_subsidy_expiration_risk.csv"
)


# ------------------------------------------------------------
# Create empty placeholder tables first
# ------------------------------------------------------------

nhpd_sd = pd.DataFrame(
    columns=[
        "NHPDPropertyID",
        "PropertyName",
        "PropertyAddress",
        "City",
        "State",
        "County",
        "PropertyStatus",
        "TotalUnits",
        "EarliestEndDate",
        "LatestEndDate",
        "jurisdiction",
        "jurisdiction_assignment_needs_review",
        "expiration_within_5_years",
    ]
)

nhpd_subsidies = pd.DataFrame(
    columns=[
        "NHPDPropertyID",
        "PropertyName",
        "jurisdiction",
        "program_name",
        "subsidy_status",
        "subsidy_end_date",
        "assisted_units",
        "subsidy_field_prefix",
        "subsidy_expiration_within_5_years",
        "assisted_units_with_expiring_subsidy",
    ]
)

nhpd_summary = pd.DataFrame(
    columns=[
        "jurisdiction",
        "federally_assisted_properties",
        "total_units_in_assisted_properties",
        "properties_expiring_within_5_years",
        "units_in_expiring_properties",
    ]
)

nhpd_subsidy_risk_summary = pd.DataFrame(
    columns=[
        "jurisdiction",
        "program_name",
        "expiring_subsidies",
        "assisted_units_with_expiring_subsidy",
    ]
)

nhpd_jurisdiction_review = pd.DataFrame()


# ============================================================
# RUN REAL NHPD PROCESSING ONLY IF DATA ARE AVAILABLE
# ============================================================

if NHPD_AVAILABLE:

    print("NHPD file found. Processing NHPD data...")

    # --------------------------------------------------------
    # Load NHPD
    # --------------------------------------------------------

    nhpd_raw = pd.read_csv(
        NHPD_RAW_PATH,
        low_memory=False,
    )

    required_nhpd_columns = [
        "NHPDPropertyID",
        "PropertyName",
        "PropertyAddress",
        "City",
        "State",
        "County",
        "PropertyStatus",
        "TotalUnits",
        "EarliestEndDate",
        "LatestEndDate",
    ]

    missing_nhpd_columns = [
        column
        for column in required_nhpd_columns
        if column not in nhpd_raw.columns
    ]

    assert not missing_nhpd_columns, (
        "Missing required NHPD columns: "
        f"{missing_nhpd_columns}"
    )


    # --------------------------------------------------------
    # Keep San Diego County properties
    # --------------------------------------------------------

    nhpd_sd = nhpd_raw[
        nhpd_raw["State"]
        .astype(str)
        .str.upper()
        .eq("CA")
        &
        nhpd_raw["County"]
        .astype(str)
        .str.contains(
            "San Diego",
            case=False,
            na=False,
        )
    ].copy()

    nhpd_sd["TotalUnits"] = pd.to_numeric(
        nhpd_sd["TotalUnits"],
        errors="coerce",
    )

    for column in [
        "EarliestEndDate",
        "LatestEndDate",
    ]:
        nhpd_sd[column] = pd.to_datetime(
            nhpd_sd[column],
            errors="coerce",
        )


    # --------------------------------------------------------
    # Assign jurisdictions
    # --------------------------------------------------------

    city_lookup = {
        city.lower(): city
        for city in SAN_DIEGO_CITIES
    }

    nhpd_sd["city_clean"] = (
        nhpd_sd["City"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    nhpd_sd["jurisdiction"] = (
        nhpd_sd["city_clean"]
        .map(city_lookup)
    )

    nhpd_sd[
        "jurisdiction_assignment_needs_review"
    ] = nhpd_sd["jurisdiction"].isna()

    # IMPORTANT:
    # Do not automatically call unmatched properties
    # "Unincorporated San Diego County".
    nhpd_sd["jurisdiction"] = (
        nhpd_sd["jurisdiction"]
        .fillna(
            "Needs jurisdiction review"
        )
    )

    nhpd_jurisdiction_review = nhpd_sd[
        nhpd_sd[
            "jurisdiction_assignment_needs_review"
        ]
    ].copy()


    # --------------------------------------------------------
    # Property-level expiration risk
    # --------------------------------------------------------

    AT_RISK_YEARS = 5

    today = pd.Timestamp.today().normalize()

    risk_cutoff = (
        today
        + pd.DateOffset(
            years=AT_RISK_YEARS
        )
    )

    nhpd_sd[
        "expiration_within_5_years"
    ] = (
        nhpd_sd["PropertyStatus"]
        .eq("Active")
        &
        nhpd_sd["EarliestEndDate"]
        .notna()
        &
        nhpd_sd["EarliestEndDate"]
        .between(
            today,
            risk_cutoff,
            inclusive="both",
        )
    )


    # --------------------------------------------------------
    # Convert NHPD program fields to subsidy-level rows
    # --------------------------------------------------------

    program_columns = [
        column
        for column in nhpd_sd.columns
        if re.fullmatch(
            r".+_\d+_ProgramName",
            column,
        )
    ]

    subsidy_records = []

    for _, row in nhpd_sd.iterrows():

        for program_column in program_columns:

            program_name = row.get(
                program_column
            )

            if pd.isna(program_name):
                continue

            prefix = program_column[
                :-len("_ProgramName")
            ]

            subsidy_records.append(
                {
                    "NHPDPropertyID": (
                        row["NHPDPropertyID"]
                    ),
                    "PropertyName": (
                        row["PropertyName"]
                    ),
                    "jurisdiction": (
                        row["jurisdiction"]
                    ),
                    "program_name": (
                        program_name
                    ),
                    "subsidy_status": (
                        row.get(
                            f"{prefix}_Status"
                        )
                    ),
                    "subsidy_end_date": (
                        row.get(
                            f"{prefix}_EndDate"
                        )
                    ),
                    "assisted_units": (
                        row.get(
                            f"{prefix}_AssistedUnits"
                        )
                    ),
                    "subsidy_field_prefix": (
                        prefix
                    ),
                }
            )

    nhpd_subsidies = pd.DataFrame(
        subsidy_records
    )

    if not nhpd_subsidies.empty:

        nhpd_subsidies[
            "subsidy_end_date"
        ] = pd.to_datetime(
            nhpd_subsidies[
                "subsidy_end_date"
            ],
            errors="coerce",
        )

        nhpd_subsidies[
            "assisted_units"
        ] = pd.to_numeric(
            nhpd_subsidies[
                "assisted_units"
            ],
            errors="coerce",
        )


        # ----------------------------------------------------
        # Subsidy-level preservation risk
        # ----------------------------------------------------

        nhpd_subsidies[
            "subsidy_expiration_within_5_years"
        ] = (
            nhpd_subsidies[
                "subsidy_end_date"
            ].notna()
            &
            nhpd_subsidies[
                "subsidy_end_date"
            ].between(
                today,
                risk_cutoff,
                inclusive="both",
            )
        )

        nhpd_subsidies[
            "assisted_units_with_expiring_subsidy"
        ] = np.where(
            nhpd_subsidies[
                "subsidy_expiration_within_5_years"
            ],
            nhpd_subsidies[
                "assisted_units"
            ],
            0,
        )


        # ----------------------------------------------------
        # Subsidy risk summary
        # ----------------------------------------------------

        nhpd_subsidy_risk_summary = (
            nhpd_subsidies[
                nhpd_subsidies[
                    "subsidy_expiration_within_5_years"
                ]
            ]
            .groupby(
                [
                    "jurisdiction",
                    "program_name",
                ],
                as_index=False,
            )
            .agg(
                expiring_subsidies=(
                    "NHPDPropertyID",
                    "count",
                ),
                assisted_units_with_expiring_subsidy=(
                    "assisted_units_with_expiring_subsidy",
                    "sum",
                ),
            )
        )


    # --------------------------------------------------------
    # Property-level jurisdiction summary
    # --------------------------------------------------------

    nhpd_active = nhpd_sd[
        nhpd_sd["PropertyStatus"]
        .eq("Active")
    ].copy()

    nhpd_active[
        "units_in_expiring_property"
    ] = np.where(
        nhpd_active[
            "expiration_within_5_years"
        ],
        nhpd_active["TotalUnits"],
        0,
    )

    nhpd_summary = (
        nhpd_active
        .groupby(
            "jurisdiction",
            as_index=False,
        )
        .agg(
            federally_assisted_properties=(
                "NHPDPropertyID",
                "nunique",
            ),
            total_units_in_assisted_properties=(
                "TotalUnits",
                "sum",
            ),
            properties_expiring_within_5_years=(
                "expiration_within_5_years",
                "sum",
            ),
            units_in_expiring_properties=(
                "units_in_expiring_property",
                "sum",
            ),
        )
    )

    print(
        "San Diego County NHPD properties:",
        nhpd_sd[
            "NHPDPropertyID"
        ].nunique(),
    )

else:

    print(
        "NHPD DATA PENDING: "
        "NHPD processing was skipped."
    )

    print(
        "Empty placeholder NHPD outputs "
        "will be created so the rest of "
        "the notebook can run."
    )


# ============================================================
# SAVE OUTPUTS IN BOTH CASES
# ============================================================

nhpd_sd.to_csv(
    nhpd_property_path,
    index=False,
)

nhpd_subsidies.to_csv(
    nhpd_subsidy_path,
    index=False,
)

nhpd_summary.to_csv(
    nhpd_summary_path,
    index=False,
)

nhpd_subsidy_risk_summary.to_csv(
    nhpd_subsidy_risk_path,
    index=False,
)


if NHPD_AVAILABLE:

    print(
        "Saved completed NHPD outputs."
    )

else:

    print(
        "Saved empty NHPD placeholder outputs."
    )

    print(
        "IMPORTANT: Empty NHPD tables mean "
        "'data pending', NOT zero assisted housing."
    )

NHPD DATA PENDING: NHPD processing was skipped.
Empty placeholder NHPD outputs will be created so the rest of the notebook can run.
Saved empty NHPD placeholder outputs.
IMPORTANT: Empty NHPD tables mean 'data pending', NOT zero assisted housing.


### Metric Dictionary

In [110]:
metric_rows = [
    # Income
    (
        "median_household_income",
        "ACS B19013",
        "B19013_001E",
        "Median household income",
        "dollars",
        "jurisdiction/countywide",
        "estimate",
    ),
    (
        "median_household_income_moe",
        "ACS B19013",
        "B19013_001M",
        "Margin of error for median household income",
        "dollars",
        "jurisdiction/countywide",
        "margin of error",
    ),
    (
        "county_median_household_income",
        "ACS B19013",
        "B19013_001E",
        "Countywide median household income used as comparison benchmark",
        "dollars",
        "countywide",
        "benchmark",
    ),
    (
        "jurisdiction_county_income_ratio",
        "ACS B19013",
        "jurisdiction MHI / countywide MHI",
        "Jurisdiction median household income divided by countywide median household income",
        "ratio",
        "jurisdiction",
        "derived",
    ),
    (
        "jurisdiction_county_income_pct",
        "ACS B19013",
        "jurisdiction_county_income_ratio * 100",
        "Jurisdiction income as a percentage of countywide median household income",
        "percent",
        "jurisdiction",
        "derived",
    ),

    # Renter burden
    (
        "renter_households_computed",
        "ACS B25070",
        "B25070_001E minus B25070_011E",
        "Renter households for which rent burden can be calculated",
        "households",
        "jurisdiction/countywide",
        "denominator",
    ),
    (
        "renter_cost_burdened_count",
        "ACS B25070",
        "B25070_007E through B25070_010E",
        "Renter households spending at least 30% of income on gross rent",
        "households",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "renter_cost_burdened_count_moe",
        "ACS B25070",
        "combined MOE",
        "Margin of error for renter cost-burdened count",
        "households",
        "jurisdiction/countywide",
        "margin of error",
    ),
    (
        "renter_cost_burdened_pct",
        "ACS B25070",
        "burdened count / computed renter households",
        "Percentage of renter households spending at least 30% of income on gross rent",
        "percent",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "renter_severely_burdened_count",
        "ACS B25070",
        "B25070_010E",
        "Renter households spending at least 50% of income on gross rent",
        "households",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "renter_severely_burdened_count_moe",
        "ACS B25070",
        "B25070_010M",
        "Margin of error for severely burdened renter households",
        "households",
        "jurisdiction/countywide",
        "margin of error",
    ),
    (
        "renter_severely_burdened_pct",
        "ACS B25070",
        "severely burdened / computed renter households",
        "Percentage of renter households spending at least 50% of income on gross rent",
        "percent",
        "jurisdiction/countywide",
        "observed burden",
    ),

    # Homeowner burden
    (
        "owner_with_mortgage_computed",
        "ACS B25091",
        "B25091_002E minus B25091_012E",
        "Homeowners with a mortgage for whom burden can be calculated",
        "households",
        "jurisdiction/countywide",
        "denominator",
    ),
    (
        "owner_without_mortgage_computed",
        "ACS B25091",
        "B25091_013E minus B25091_023E",
        "Homeowners without a mortgage for whom burden can be calculated",
        "households",
        "jurisdiction/countywide",
        "denominator",
    ),
    (
        "owner_cost_burdened_pct",
        "ACS B25091",
        "owner burdened count / computed owner households",
        "Percentage of homeowner households spending at least 30% of income on owner costs",
        "percent",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "owner_severely_burdened_pct",
        "ACS B25091",
        "severely burdened owners / computed owner households",
        "Percentage of homeowner households spending at least 50% of income on owner costs",
        "percent",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "owner_with_mortgage_burdened_pct",
        "ACS B25091",
        "with-mortgage burdened / with-mortgage computed",
        "Cost-burden percentage for homeowners with a mortgage",
        "percent",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "owner_with_mortgage_severely_burdened_pct",
        "ACS B25091",
        "B25091_011E / with-mortgage computed",
        "Severe cost-burden percentage for homeowners with a mortgage",
        "percent",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "owner_without_mortgage_burdened_pct",
        "ACS B25091",
        "without-mortgage burdened / without-mortgage computed",
        "Cost-burden percentage for homeowners without a mortgage",
        "percent",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "owner_without_mortgage_severely_burdened_pct",
        "ACS B25091",
        "B25091_022E / without-mortgage computed",
        "Severe cost-burden percentage for homeowners without a mortgage",
        "percent",
        "jurisdiction/countywide",
        "observed burden",
    ),

        (
        "owner_households_computed",
        "ACS B25091",
        "owner_with_mortgage_computed + owner_without_mortgage_computed",
        "Homeowner households for which housing-cost burden can be calculated",
        "households",
        "jurisdiction/countywide",
        "denominator",
    ),
    (
        "owner_cost_burdened_count",
        "ACS B25091",
        "B25091 burden categories at or above 30%",
        "Homeowner households spending at least 30% of income on selected monthly owner costs",
        "households",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "owner_cost_burdened_count_moe",
        "ACS B25091",
        "combined margin of error",
        "Margin of error for cost-burdened homeowner count",
        "households",
        "jurisdiction/countywide",
        "margin of error",
    ),
    (
        "owner_severely_burdened_count",
        "ACS B25091",
        "B25091_011E + B25091_022E",
        "Homeowner households spending at least 50% of income on selected monthly owner costs",
        "households",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "owner_severely_burdened_count_moe",
        "ACS B25091",
        "combined B25091_011M and B25091_022M",
        "Margin of error for severely cost-burdened homeowner count",
        "households",
        "jurisdiction/countywide",
        "margin of error",
    ),
    (
        "owner_with_mortgage_burdened_count",
        "ACS B25091",
        "with-mortgage burden categories at or above 30%",
        "Cost-burdened homeowner households with a mortgage",
        "households",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "owner_with_mortgage_severely_burdened_count",
        "ACS B25091",
        "B25091_011E",
        "Severely cost-burdened homeowner households with a mortgage",
        "households",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "owner_without_mortgage_burdened_count",
        "ACS B25091",
        "without-mortgage burden categories at or above 30%",
        "Cost-burdened homeowner households without a mortgage",
        "households",
        "jurisdiction/countywide",
        "observed burden",
    ),
    (
        "owner_without_mortgage_severely_burdened_count",
        "ACS B25091",
        "B25091_022E",
        "Severely cost-burdened homeowner households without a mortgage",
        "households",
        "jurisdiction/countywide",
        "observed burden",
    ),

    # Rent benchmark
    (
        "median_gross_rent_1br",
        "ACS B25031",
        "B25031_003E",
        "Median monthly gross rent for a one-bedroom renter-occupied unit",
        "dollars/month",
        "jurisdiction/countywide",
        "estimate",
    ),
    (
        "median_gross_rent_1br_moe",
        "ACS B25031",
        "B25031_003M",
        "Margin of error for one-bedroom median gross rent",
        "dollars/month",
        "jurisdiction/countywide",
        "margin of error",
    ),
    (
        "one_br_rent_to_income_pct",
        "ACS B25031 + B19013",
        "annual one-bedroom rent / median household income",
        "Benchmark comparing one-bedroom gross rent with median income for all households; not observed renter burden",
        "percent",
        "jurisdiction/countywide",
        "benchmark",
    ),

    # Owner-cost benchmark
    (
        "median_owner_cost_with_mortgage",
        "ACS B25088",
        "B25088_002E",
        "Median selected monthly owner costs for owners with a mortgage",
        "dollars/month",
        "jurisdiction/countywide",
        "estimate",
    ),
    (
        "median_owner_cost_without_mortgage",
        "ACS B25088",
        "B25088_003E",
        "Median selected monthly owner costs for owners without a mortgage",
        "dollars/month",
        "jurisdiction/countywide",
        "estimate",
    ),
        (
        "median_owner_cost_with_mortgage_moe",
        "ACS B25088",
        "B25088_002M",
        "Margin of error for median selected monthly owner costs with a mortgage",
        "dollars/month",
        "jurisdiction/countywide",
        "margin of error",
    ),
    (
        "median_owner_cost_without_mortgage_moe",
        "ACS B25088",
        "B25088_003M",
        "Margin of error for median selected monthly owner costs without a mortgage",
        "dollars/month",
        "jurisdiction/countywide",
        "margin of error",
    ),
    (
        "median_owner_cost_with_mortgage_top_coded",
        "ACS B25088",
        "B25088_002EA",
        "Indicates that the median falls in the highest open-ended owner-cost interval",
        "boolean",
        "jurisdiction/countywide",
        "data-quality flag",
    ),
    (
        "median_owner_cost_without_mortgage_top_coded",
        "ACS B25088",
        "B25088_003EA",
        "Indicates that the median falls in the highest open-ended owner-cost interval",
        "boolean",
        "jurisdiction/countywide",
        "data-quality flag",
    ),
    (
        "owner_cost_with_mortgage_to_income_pct",
        "ACS B25088 + B19013",
        "annual owner costs / median household income",
        "Benchmark using median household income for all households; not observed homeowner burden",
        "percent",
        "jurisdiction/countywide",
        "benchmark",
    ),
    (
        "owner_cost_without_mortgage_to_income_pct",
        "ACS B25088 + B19013",
        "annual owner costs / median household income",
        "Benchmark using median household income for all households; not observed homeowner burden",
        "percent",
        "jurisdiction/countywide",
        "benchmark",
    ),

    # HUD
    (
        "hud_four_person_median_family_income",
        "HUD FY2026 Income Limits",
        "four-person HUD median family income",
        "HUD regional four-person median family income benchmark",
        "dollars/year",
        "San Diego regional",
        "benchmark",
    ),
    (
        "hud_four_person_extremely_low_income_limit",
        "HUD FY2026 Income Limits",
        "four-person extremely low-income limit",
        "HUD four-person extremely low-income limit",
        "dollars/year",
        "San Diego regional",
        "benchmark",
    ),
    (
        "hud_four_person_very_low_income_limit",
        "HUD FY2026 Income Limits",
        "four-person very low-income limit",
        "HUD four-person very low-income limit",
        "dollars/year",
        "San Diego regional",
        "benchmark",
    ),
    (
        "hud_four_person_low_income_limit",
        "HUD FY2026 Income Limits",
        "four-person low-income limit",
        "HUD four-person low-income limit",
        "dollars/year",
        "San Diego regional",
        "benchmark",
    ),

    # NHPD
    (
        "federally_assisted_properties",
        "NHPD",
        "NHPDPropertyID",
        "Number of active federally assisted properties",
        "properties",
        "jurisdiction",
        "inventory",
    ),
    (
        "total_units_in_assisted_properties",
        "NHPD",
        "TotalUnits",
        "Total units located in active assisted properties; includes assisted and unassisted units",
        "housing units",
        "jurisdiction",
        "inventory",
    ),
    (
        "properties_expiring_within_5_years",
        "NHPD",
        "PropertyStatus + EarliestEndDate",
        "Active assisted properties with an earliest subsidy expiration within five years",
        "properties",
        "jurisdiction",
        "preservation risk",
    ),
    (
        "units_in_expiring_properties",
        "NHPD",
        "TotalUnits + expiration flag",
        "Total units located in properties with an earliest subsidy expiration within five years",
        "housing units",
        "jurisdiction",
        "preservation risk",
    ),
        (
        "expiring_subsidies",
        "NHPD",
        "subsidy_end_date",
        "Number of subsidy records with an expiration date within the next five years",
        "subsidies",
        "jurisdiction/program",
        "preservation risk",
    ),
    (
        "assisted_units_with_expiring_subsidy",
        "NHPD",
        "assisted_units + subsidy_end_date",
        "Assisted units reported on subsidy records expiring within five years; may overlap across subsidy programs",
        "assisted units",
        "jurisdiction/program",
        "preservation risk",
    ),
]

metric_dictionary = pd.DataFrame(
    metric_rows,
    columns=[
        "output_metric",
        "source_table",
        "source_variables",
        "definition",
        "unit",
        "geography",
        "metric_type",
    ],
)

dictionary_path = (
    DOCS_DIR
    / "housing_need_affordability_metric_dictionary.csv"
)

metric_dictionary.to_csv(
    dictionary_path,
    index=False,
)

display(metric_dictionary)

print("Saved metric dictionary:")
print(dictionary_path)

,output_metric,source_table,source_variables,definition,unit,geography,metric_type
0,median_household_income,ACS B19013,B19013_001E,Median household income,dollars,jurisdiction/countywide,estimate
1,median_household_income_moe,ACS B19013,B19013_001M,Margin of error for median household income,dollars,jurisdiction/countywide,margin of error
2,county_median_household_income,ACS B19013,B19013_001E,Countywide median household income used as com...,dollars,countywide,benchmark
3,jurisdiction_county_income_ratio,ACS B19013,jurisdiction MHI / countywide MHI,Jurisdiction median household income divided b...,ratio,jurisdiction,derived
4,jurisdiction_county_income_pct,ACS B19013,jurisdiction_county_income_ratio * 100,Jurisdiction income as a percentage of countyw...,percent,jurisdiction,derived
5,renter_households_computed,ACS B25070,B25070_001E minus B25070_011E,Renter households for which rent burden can be...,households,jurisdiction/countywide,denominator
6,renter_cost_burdened_count,ACS B25070,B25070_007E through B25070_010E,Renter households spending at least 30% of inc...,households,jurisdiction/countywide,observed burden
7,renter_cost_burdened_count_moe,ACS B25070,combined MOE,Margin of error for renter cost-burdened count,households,jurisdiction/countywide,margin of error
8,renter_cost_burdened_pct,ACS B25070,burdened count / computed renter households,Percentage of renter households spending at le...,percent,jurisdiction/countywide,observed burden
9,renter_severely_burdened_count,ACS B25070,B25070_010E,Renter households spending at least 50% of inc...,households,jurisdiction/countywide,observed burden


Saved metric dictionary:
/Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work/docs/housing_need_affordability_metric_dictionary.csv


## Limitations and Next Steps

- ACS values are estimates and include margins of error.
- The 2024 ACS five-year release represents the 2020–2024 period.
- The current jurisdiction output includes the 18 incorporated cities and a separate countywide record.
- The unincorporated county area is not treated as an ACS place and requires a separate method.
- Median household income is different from median family income and HUD Area Median Income.
- The one-bedroom rent-to-income measure is a benchmark and is not the same as observed renter cost burden.
- HUD income limits are regional and should not be presented as jurisdiction-specific median incomes.
- NHPD federally assisted housing and preservation-risk data are included. Property-level TotalUnits include both assisted and unassisted units, so subsidy-level assisted-unit counts are kept separate.
- Five-year preservation risk identifies active properties whose earliest known subsidy expiration occurs within the next five years.
- NHPD jurisdiction assignment for properties outside the 18 incorporated city names should be boundary-verified before property-level mapping.
- HUD CHAS is not currently included and may be added later if additional household-level affordability detail is needed.

## Data Validation

### Select records for detailed validation

Selects the San Diego County countywide record, San Diego, Del Mar, and Lemon Grove for detailed validation. This provides examples from the county, a large city, and smaller cities, but it is not a separate external review of every jurisdiction.

In [111]:
VALIDATION_JURISDICTIONS = [
    "San Diego County (Countywide)",
    "San Diego",
    "Del Mar",
    "Lemon Grove",
]

validation_output = affordability[
    affordability["jurisdiction"].isin(
        VALIDATION_JURISDICTIONS
    )
].copy()

validation_output = validation_output.sort_values(
    "jurisdiction"
).reset_index(drop=True)

display(
    validation_output[
        [
            "jurisdiction",
            "geography_type",
            "median_household_income",
            "jurisdiction_county_income_ratio",
            "renter_cost_burdened_pct",
            "renter_severely_burdened_pct",
            "owner_cost_burdened_pct",
            "owner_severely_burdened_pct",
            "median_gross_rent_1br",
            "one_br_rent_to_income_pct",
            "median_owner_cost_with_mortgage",
            "median_owner_cost_without_mortgage",
        ]
    ]
)

,jurisdiction,geography_type,median_household_income,jurisdiction_county_income_ratio,renter_cost_burdened_pct,renter_severely_burdened_pct,owner_cost_burdened_pct,owner_severely_burdened_pct,median_gross_rent_1br,one_br_rent_to_income_pct,median_owner_cost_with_mortgage,median_owner_cost_without_mortgage
0,Del Mar,jurisdiction,199152.0,1.874,45.47,27.92,26.24,15.91,3003.0,18.09,4001.0,1208.0
1,Lemon Grove,jurisdiction,88009.0,0.828,61.82,32.20,39.20,18.63,1379.0,18.80,2538.0,615.0
2,San Diego,jurisdiction,108077.0,1.017,54.68,27.16,31.00,13.76,1963.0,21.80,3252.0,856.0
3,San Diego County (Countywide),county,106268.0,1.000,57.79,29.56,32.15,14.28,1888.0,21.32,3184.0,871.0


### Recalculate sample metrics from raw data

In [112]:
# Confirm that the earlier geography-cleaning cell has been run.
if "geographies" not in globals():
    raise NameError(
        "Run the earlier cell that creates `geographies` before this validation cell."
    )

raw_validation = geographies[
    geographies["jurisdiction"].isin(
        VALIDATION_JURISDICTIONS
    )
].copy()

print("Validation rows found:", len(raw_validation))

display(
    raw_validation[
        [
            "jurisdiction",
            "geography_type",
        ]
    ]
)

# Median household income
raw_validation[
    "check_median_household_income"
] = raw_validation["B19013_001E"]

county_income_check = raw_validation.loc[
    raw_validation["jurisdiction"].eq(
        "San Diego County (Countywide)"
    ),
    "B19013_001E",
].iloc[0]

raw_validation[
    "check_income_ratio"
] = (
    raw_validation["B19013_001E"]
    / county_income_check
).round(3)


# Renter burden
raw_validation[
    "check_renter_computed"
] = (
    raw_validation["B25070_001E"]
    - raw_validation["B25070_011E"]
)

raw_validation[
    "check_renter_burdened_count"
] = raw_validation[
    [
        "B25070_007E",
        "B25070_008E",
        "B25070_009E",
        "B25070_010E",
    ]
].sum(axis=1)

raw_validation[
    "check_renter_severe_count"
] = raw_validation["B25070_010E"]

raw_validation[
    "check_renter_burdened_pct"
] = (
    raw_validation[
        "check_renter_burdened_count"
    ]
    / raw_validation["check_renter_computed"]
    * 100
).round(2)

raw_validation[
    "check_renter_severe_pct"
] = (
    raw_validation[
        "check_renter_severe_count"
    ]
    / raw_validation["check_renter_computed"]
    * 100
).round(2)


# Homeowner burden
raw_validation[
    "check_owner_computed"
] = (
    raw_validation["B25091_002E"]
    - raw_validation["B25091_012E"]
    + raw_validation["B25091_013E"]
    - raw_validation["B25091_023E"]
)

raw_validation[
    "check_owner_burdened_count"
] = raw_validation[
    [
        "B25091_008E",
        "B25091_009E",
        "B25091_010E",
        "B25091_011E",
        "B25091_019E",
        "B25091_020E",
        "B25091_021E",
        "B25091_022E",
    ]
].sum(axis=1)

raw_validation[
    "check_owner_severe_count"
] = (
    raw_validation["B25091_011E"]
    + raw_validation["B25091_022E"]
)

raw_validation[
    "check_owner_burdened_pct"
] = (
    raw_validation[
        "check_owner_burdened_count"
    ]
    / raw_validation["check_owner_computed"]
    * 100
).round(2)

raw_validation[
    "check_owner_severe_pct"
] = (
    raw_validation[
        "check_owner_severe_count"
    ]
    / raw_validation["check_owner_computed"]
    * 100
).round(2)


# Rent and ownership costs
raw_validation[
    "check_median_gross_rent_1br"
] = raw_validation["B25031_003E"]

raw_validation[
    "check_one_br_rent_to_income_pct"
] = (
    raw_validation["B25031_003E"]
    * 12
    / raw_validation["B19013_001E"]
    * 100
).round(2)

raw_validation[
    "check_owner_cost_with_mortgage"
] = raw_validation["B25088_002E"]

raw_validation[
    "check_owner_cost_without_mortgage"
] = raw_validation["B25088_003E"]

display(
    raw_validation[
        [
            "jurisdiction",
            "check_median_household_income",
            "check_income_ratio",
            "check_renter_burdened_pct",
            "check_renter_severe_pct",
            "check_owner_burdened_pct",
            "check_owner_severe_pct",
            "check_median_gross_rent_1br",
            "check_one_br_rent_to_income_pct",
            "check_owner_cost_with_mortgage",
            "check_owner_cost_without_mortgage",
        ]
    ].sort_values("jurisdiction")
)

Validation rows found: 4


,jurisdiction,geography_type
0,San Diego County (Countywide),county
4,Del Mar,jurisdiction
10,Lemon Grove,jurisdiction
14,San Diego,jurisdiction


,jurisdiction,check_median_household_income,check_income_ratio,check_renter_burdened_pct,check_renter_severe_pct,check_owner_burdened_pct,check_owner_severe_pct,check_median_gross_rent_1br,check_one_br_rent_to_income_pct,check_owner_cost_with_mortgage,check_owner_cost_without_mortgage
4,Del Mar,199152.0,1.874,45.47,27.92,26.24,15.91,3003.0,18.09,4001.0,1208.0
10,Lemon Grove,88009.0,0.828,61.82,32.20,39.20,18.63,1379.0,18.80,2538.0,615.0
14,San Diego,108077.0,1.017,54.68,27.16,31.00,13.76,1963.0,21.80,3252.0,856.0
0,San Diego County (Countywide),106268.0,1.000,57.79,29.56,32.15,14.28,1888.0,21.32,3184.0,871.0


### Compare processed and recalculated values

In [113]:
processed_check = affordability[
    affordability["jurisdiction"].isin(
        VALIDATION_JURISDICTIONS
    )
][
    [
        "jurisdiction",
        "median_household_income",
        "jurisdiction_county_income_ratio",
        "renter_cost_burdened_pct",
        "renter_severely_burdened_pct",
        "owner_cost_burdened_pct",
        "owner_severely_burdened_pct",
        "median_gross_rent_1br",
        "one_br_rent_to_income_pct",
        "median_owner_cost_with_mortgage",
        "median_owner_cost_without_mortgage",
    ]
].copy()

raw_check = raw_validation[
    [
        "jurisdiction",
        "check_median_household_income",
        "check_income_ratio",
        "check_renter_burdened_pct",
        "check_renter_severe_pct",
        "check_owner_burdened_pct",
        "check_owner_severe_pct",
        "check_median_gross_rent_1br",
        "check_one_br_rent_to_income_pct",
        "check_owner_cost_with_mortgage",
        "check_owner_cost_without_mortgage",
    ]
].copy()

comparison = processed_check.merge(
    raw_check,
    on="jurisdiction",
    how="outer",
    validate="one_to_one",
)

comparison["income_matches"] = np.isclose(
    comparison["median_household_income"],
    comparison["check_median_household_income"],
    equal_nan=True,
)

comparison["income_ratio_matches"] = np.isclose(
    comparison["jurisdiction_county_income_ratio"],
    comparison["check_income_ratio"],
    equal_nan=True,
)

comparison["renter_burden_matches"] = np.isclose(
    comparison["renter_cost_burdened_pct"],
    comparison["check_renter_burdened_pct"],
    equal_nan=True,
)

comparison["severe_renter_burden_matches"] = np.isclose(
    comparison["renter_severely_burdened_pct"],
    comparison["check_renter_severe_pct"],
    equal_nan=True,
)

comparison["owner_burden_matches"] = np.isclose(
    comparison["owner_cost_burdened_pct"],
    comparison["check_owner_burdened_pct"],
    equal_nan=True,
)

comparison["severe_owner_burden_matches"] = np.isclose(
    comparison["owner_severely_burdened_pct"],
    comparison["check_owner_severe_pct"],
    equal_nan=True,
)

comparison["one_br_rent_matches"] = np.isclose(
    comparison["median_gross_rent_1br"],
    comparison["check_median_gross_rent_1br"],
    equal_nan=True,
)

comparison["rent_to_income_matches"] = np.isclose(
    comparison["one_br_rent_to_income_pct"],
    comparison[
        "check_one_br_rent_to_income_pct"
    ],
    equal_nan=True,
)

comparison["owner_cost_with_mortgage_matches"] = (
    np.isclose(
        comparison[
            "median_owner_cost_with_mortgage"
        ],
        comparison[
            "check_owner_cost_with_mortgage"
        ],
        equal_nan=True,
    )
)

comparison["owner_cost_without_mortgage_matches"] = (
    np.isclose(
        comparison[
            "median_owner_cost_without_mortgage"
        ],
        comparison[
            "check_owner_cost_without_mortgage"
        ],
        equal_nan=True,
    )
)

match_columns = [
    column
    for column in comparison.columns
    if column.endswith("_matches")
]

comparison["all_metrics_match"] = (
    comparison[match_columns].all(axis=1)
)

display(
    comparison[
        [
            "jurisdiction",
            *match_columns,
            "all_metrics_match",
        ]
    ]
)

,jurisdiction,income_matches,income_ratio_matches,renter_burden_matches,severe_renter_burden_matches,owner_burden_matches,severe_owner_burden_matches,one_br_rent_matches,rent_to_income_matches,owner_cost_with_mortgage_matches,owner_cost_without_mortgage_matches,all_metrics_match
0,Del Mar,True,True,True,True,True,True,True,True,True,True,True
1,Lemon Grove,True,True,True,True,True,True,True,True,True,True,True
2,San Diego,True,True,True,True,True,True,True,True,True,True,True
3,San Diego County (Countywide),True,True,True,True,True,True,True,True,True,True,True


### Confirm the calculations match

In [114]:
failed_checks = comparison.loc[
    ~comparison["all_metrics_match"],
    [
        "jurisdiction",
        *match_columns,
    ],
]

if failed_checks.empty:
    print(
        "PASS: All selected processed metrics match "
        "the values recalculated from the raw ACS fields."
    )
else:
    display(failed_checks)

    raise AssertionError(
        "One or more processed metrics do not match "
        "the raw ACS calculations."
    )

PASS: All selected processed metrics match the values recalculated from the raw ACS fields.


### Check dataset structure

In [115]:
required_columns = [
    "acs_year",
    "acs_period",
    "geoid",
    "jurisdiction",
    "geography_type",
    "median_household_income",
    "median_household_income_moe",
    "county_median_household_income",
    "jurisdiction_county_income_ratio",
    "renter_households_computed",
    "renter_cost_burdened_count",
    "renter_cost_burdened_pct",
    "renter_severely_burdened_count",
    "renter_severely_burdened_pct",
    "owner_households_computed",
    "owner_with_mortgage_computed",
    "owner_without_mortgage_computed",

    "owner_with_mortgage_burdened_count",
    "owner_with_mortgage_burdened_pct",
    "owner_with_mortgage_severely_burdened_count",
    "owner_with_mortgage_severely_burdened_pct",

    "owner_without_mortgage_burdened_count",
    "owner_without_mortgage_burdened_pct",
    "owner_without_mortgage_severely_burdened_count",
    "owner_without_mortgage_severely_burdened_pct",
    "owner_cost_burdened_count",
    "owner_cost_burdened_pct",
    "owner_severely_burdened_count",
    "owner_severely_burdened_pct",
    "median_gross_rent_1br",
    "one_br_rent_to_income_pct",
    "median_owner_cost_with_mortgage",
    "median_owner_cost_without_mortgage",
    "source",
]

missing_columns = [
    column
    for column in required_columns
    if column not in affordability.columns
]

duplicate_geoids = affordability[
    affordability["geoid"].duplicated(keep=False)
]

duplicate_jurisdictions = affordability[
    affordability["jurisdiction"].duplicated(
        keep=False
    )
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

assert duplicate_geoids.empty, (
    "Duplicate GEOIDs found:\n"
    f"{duplicate_geoids[['geoid', 'jurisdiction']]}"
)

assert duplicate_jurisdictions.empty, (
    "Duplicate jurisdictions found:\n"
    f"{duplicate_jurisdictions[['geoid', 'jurisdiction']]}"
)

assert len(
    affordability[
        affordability[
            "geography_type"
        ].eq("jurisdiction")
    ]
) == 18

assert len(
    affordability[
        affordability[
            "geography_type"
        ].eq("county")
    ]
) == 1

print("PASS: All required columns are present.")
print("PASS: GEOIDs are unique.")
print("PASS: Jurisdiction names are unique.")
print("PASS: 18 cities and 1 county record are present.")

PASS: All required columns are present.
PASS: GEOIDs are unique.
PASS: Jurisdiction names are unique.
PASS: 18 cities and 1 county record are present.


### Check for invalid values

In [116]:
percentage_columns = [
    "renter_cost_burdened_pct",
    "renter_severely_burdened_pct",
    "owner_cost_burdened_pct",
    "owner_severely_burdened_pct",
    "owner_with_mortgage_burdened_pct",
    "owner_with_mortgage_severely_burdened_pct",
    "owner_without_mortgage_burdened_pct",
    "owner_without_mortgage_severely_burdened_pct",
    "one_br_rent_to_income_pct",
    "owner_cost_with_mortgage_to_income_pct",
    "owner_cost_without_mortgage_to_income_pct",
]

quality_issues = []

for column in percentage_columns:
    invalid_rows = affordability.loc[
        affordability[column].notna()
        & ~affordability[column].between(0, 100),
        ["jurisdiction", column],
    ]

    if not invalid_rows.empty:
        invalid_rows = invalid_rows.copy()
        invalid_rows["issue"] = (
            f"{column} is outside 0–100"
        )

        quality_issues.append(invalid_rows)

negative_value_columns = [
    "median_household_income",
    "renter_households_computed",
    "renter_cost_burdened_count",
    "renter_severely_burdened_count",
    "owner_households_computed",
    "owner_cost_burdened_count",
    "owner_severely_burdened_count",
    "median_gross_rent_1br",
    "median_owner_cost_with_mortgage",
    "median_owner_cost_without_mortgage",
]

for column in negative_value_columns:
    invalid_rows = affordability.loc[
        affordability[column].notna()
        & affordability[column].lt(0),
        ["jurisdiction", column],
    ]

    if not invalid_rows.empty:
        invalid_rows = invalid_rows.copy()
        invalid_rows["issue"] = (
            f"{column} is negative"
        )

        quality_issues.append(invalid_rows)

if quality_issues:
    quality_issue_report = pd.concat(
        quality_issues,
        ignore_index=True,
        sort=False,
    )

    display(quality_issue_report)

else:
    quality_issue_report = pd.DataFrame()

    print(
        "PASS: No percentages outside 0–100 "
        "and no negative metric values were found."
    )

PASS: No percentages outside 0–100 and no negative metric values were found.


### Check logical consistency

In [117]:
logical_checks = pd.DataFrame(
    {
        "jurisdiction": affordability[
            "jurisdiction"
        ],
        "severe_renter_not_above_total": (
            affordability[
                "renter_severely_burdened_count"
            ]
            <= affordability[
                "renter_cost_burdened_count"
            ]
        ),
        "severe_owner_not_above_total": (
            affordability[
                "owner_severely_burdened_count"
            ]
            <= affordability[
                "owner_cost_burdened_count"
            ]
        ),
        "renter_burden_not_above_denominator": (
            affordability[
                "renter_cost_burdened_count"
            ]
            <= affordability[
                "renter_households_computed"
            ]
        ),
        "owner_burden_not_above_denominator": (
            affordability[
                "owner_cost_burdened_count"
            ]
            <= affordability[
                "owner_households_computed"
            ]
        ),
    }
)

logical_check_columns = [
    column
    for column in logical_checks.columns
    if column != "jurisdiction"
]

logical_checks["all_logical_checks_pass"] = (
    logical_checks[
        logical_check_columns
    ].all(axis=1)
)

failed_logical_checks = logical_checks.loc[
    ~logical_checks["all_logical_checks_pass"]
]

if failed_logical_checks.empty:
    print("PASS: All logical consistency checks passed.")

else:
    display(failed_logical_checks)

    raise AssertionError(
        "One or more logical consistency checks failed."
    )

PASS: All logical consistency checks passed.


### Validate HUD income limits

Checks that the HUD values are complete, numeric, and positive. It also confirms that extremely low-income, very low-income, and low-income limits appear in the correct order. HUD may adjust these limits, so the low-income limit is not required to be below the reported median family income.

In [118]:
required_hud_columns = [
    "hud_fiscal_year",
    "effective_date",
    "hud_area_name",
    "county_name",
    "four_person_hud_median_family_income",
    "four_person_extremely_low_income_limit",
    "four_person_very_low_income_limit",
    "four_person_low_income_limit",
]

missing_hud_columns = [
    column
    for column in required_hud_columns
    if column not in hud_output.columns
]

assert not missing_hud_columns, (
    f"Missing HUD columns: {missing_hud_columns}"
)

assert len(hud_output) == 1, (
    f"Expected one San Diego HUD record, found {len(hud_output)}."
)

hud_values = hud_output.loc[
    0,
    [
        "four_person_hud_median_family_income",
        "four_person_extremely_low_income_limit",
        "four_person_very_low_income_limit",
        "four_person_low_income_limit",
    ],
].apply(pd.to_numeric, errors="coerce")

assert hud_values.notna().all(), (
    "One or more HUD values are missing or nonnumeric."
)

assert (hud_values > 0).all(), (
    "One or more HUD values are zero or negative."
)

median_income = hud_values[
    "four_person_hud_median_family_income"
]

extremely_low_limit = hud_values[
    "four_person_extremely_low_income_limit"
]

very_low_limit = hud_values[
    "four_person_very_low_income_limit"
]

low_income_limit = hud_values[
    "four_person_low_income_limit"
]

# The three program limits should maintain their correct order.
assert (
    extremely_low_limit
    <= very_low_limit
    <= low_income_limit
), (
    "HUD income limits are not in the expected order: "
    "extremely low <= very low <= low income."
)

print("PASS: HUD four-person values are present and numeric.")
print("PASS: HUD income limits follow the expected category order.")
print()
print(f"HUD median family income: ${median_income:,.0f}")
print(f"Extremely low-income limit: ${extremely_low_limit:,.0f}")
print(f"Very low-income limit: ${very_low_limit:,.0f}")
print(f"Low-income limit: ${low_income_limit:,.0f}")
print()
print(
    "Note: HUD program limits may differ from simple percentages "
    "of median family income because HUD applies methodological adjustments."
)

display(hud_output)

PASS: HUD four-person values are present and numeric.
PASS: HUD income limits follow the expected category order.

HUD median family income: $130,900
Extremely low-income limit: $52,450
Very low-income limit: $87,450
Low-income limit: $139,900

Note: HUD program limits may differ from simple percentages of median family income because HUD applies methodological adjustments.


,hud_fiscal_year,effective_date,geography_type,hud_area_name,county_name,four_person_hud_median_family_income,four_person_extremely_low_income_limit,four_person_very_low_income_limit,four_person_low_income_limit,source,source_file,note
0,2026,2026-05-01,HUD income-limit area,"San Diego-Chula Vista-Carlsbad, CA MSA",San Diego County,130900,52450,87450,139900,HUD FY 2026 Section 8 Income Limits,Section8-FY26.xlsx,"HUD AMI is a regional income-limit benchmark, ..."


### Fresh Census API spot-checks

Makes fresh Census API requests for selected metrics and compares them with the processed dataset. This confirms that selected processed values match the current published ACS API values.

The corresponding data.census.gov table URL is also saved for optional manual review.

In [119]:
EXTERNAL_SPOT_CHECKS = [
    {
        "jurisdiction": "San Diego",
        "table": "B19013",
        "variable": "B19013_001E",
        "processed_column": "median_household_income",
    },
    {
        "jurisdiction": "Del Mar",
        "table": "B25031",
        "variable": "B25031_003E",
        "processed_column": "median_gross_rent_1br",
    },
    {
        "jurisdiction": "Lemon Grove",
        "table": "B25088",
        "variable": "B25088_002E",
        "processed_column": "median_owner_cost_with_mortgage",
    },
]

place_codes = (
    sd_places
    .set_index("jurisdiction")["place"]
    .to_dict()
)

external_rows = []

for check in EXTERNAL_SPOT_CHECKS:

    jurisdiction = (
        check["jurisdiction"]
    )

    place_code = (
        place_codes[jurisdiction]
    )

    rows = get_json(
        ACS_BASE_URL,
        params={
            "get": (
                f"NAME,{check['variable']}"
            ),
            "for": (
                f"place:{place_code}"
            ),
            "in": (
                f"state:{CALIFORNIA_STATE_FIPS}"
            ),
            "key": CENSUS_API_KEY,
        },
    )

    census_value = pd.to_numeric(
        rows[1][1],
        errors="coerce",
    )

    processed_value = (
        affordability.loc[
            affordability[
                "jurisdiction"
            ].eq(jurisdiction),
            check[
                "processed_column"
            ],
        ].iloc[0]
    )

    external_rows.append(
        {
            "jurisdiction": jurisdiction,
            "table": check["table"],
            "variable": check["variable"],
            "processed_column": (
                check["processed_column"]
            ),
            "processed_value": (
                processed_value
            ),
            "fresh_census_value": (
                census_value
            ),
            "matches": np.isclose(
                processed_value,
                census_value,
                equal_nan=True,
            ),
            "data_census_table": (
                "https://data.census.gov/"
                "table/"
                f"ACSDT5Y{ACS_YEAR}."
                f"{check['table']}"
            ),
        }
    )

external_spot_checks = (
    pd.DataFrame(
        external_rows
    )
)

assert (
    external_spot_checks[
        "matches"
    ].all()
), (
    "One or more external "
    "Census spot-checks failed."
)

external_validation_path = (
    PROCESSED_DIR
    / "housing_need_affordability_external_validation.csv"
)

external_spot_checks.to_csv(
    external_validation_path,
    index=False,
)

display(external_spot_checks)

print(
    "PASS: External Census "
    "spot-checks matched."
)

,jurisdiction,table,variable,processed_column,processed_value,fresh_census_value,matches,data_census_table
0,San Diego,B19013,B19013_001E,median_household_income,108077.0,108077,True,https://data.census.gov/table/ACSDT5Y2024.B19013
1,Del Mar,B25031,B25031_003E,median_gross_rent_1br,3003.0,3003,True,https://data.census.gov/table/ACSDT5Y2024.B25031
2,Lemon Grove,B25088,B25088_002E,median_owner_cost_with_mortgage,2538.0,2538,True,https://data.census.gov/table/ACSDT5Y2024.B25088


PASS: External Census spot-checks matched.


### Save the validation report

In [120]:
validation_report = comparison[
    [
        "jurisdiction",
        *match_columns,
        "all_metrics_match",
    ]
].copy()

validation_report["validated_against"] = (
    "Raw 2024 ACS five-year variables"
)

validation_report["validation_date"] = (
    pd.Timestamp.today().date().isoformat()
)

validation_report_path = (
    PROCESSED_DIR
    / "housing_need_affordability_validation_report.csv"
)

validation_report.to_csv(
    validation_report_path,
    index=False,
)

print("Saved validation report:")
print(validation_report_path)

display(validation_report)

Saved validation report:
/Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work/data/processed/housing_need_affordability_validation_report.csv


,jurisdiction,income_matches,income_ratio_matches,renter_burden_matches,severe_renter_burden_matches,owner_burden_matches,severe_owner_burden_matches,one_br_rent_matches,rent_to_income_matches,owner_cost_with_mortgage_matches,owner_cost_without_mortgage_matches,all_metrics_match,validated_against,validation_date
0,Del Mar,True,True,True,True,True,True,True,True,True,True,True,Raw 2024 ACS five-year variables,2026-08-11
1,Lemon Grove,True,True,True,True,True,True,True,True,True,True,True,Raw 2024 ACS five-year variables,2026-08-11
2,San Diego,True,True,True,True,True,True,True,True,True,True,True,Raw 2024 ACS five-year variables,2026-08-11
3,San Diego County (Countywide),True,True,True,True,True,True,True,True,True,True,True,Raw 2024 ACS five-year variables,2026-08-11


### Confirm all output files

In [121]:
expected_files = [
    acs_output_path,
    hud_output_path,
    combined_output_path,
    geography_dimension_path,
    dictionary_path,
    nhpd_property_path,
    nhpd_subsidy_path,
    nhpd_summary_path,
    validation_report_path,
    external_validation_path,
    nhpd_subsidy_risk_path,
]

print("FINAL OUTPUTS\n")

for file_path in expected_files:

    status = (
        "FOUND"
        if file_path.exists()
        else "MISSING"
    )

    print(
        f"{status}: "
        f"{file_path.relative_to(ROOT)}"
    )

assert all(
    file_path.exists()
    for file_path in expected_files
)

print(
    "\nAll expected output files "
    "were created."
)

FINAL OUTPUTS

FOUND: data/processed/acs_2024_housing_need_affordability_by_jurisdiction.csv
FOUND: data/processed/hud_2026_four_person_income_limits_san_diego.csv
FOUND: data/processed/housing_need_affordability_acs2024_hud2026.csv
FOUND: data/processed/housing_dashboard_geography_dimension.csv
FOUND: docs/housing_need_affordability_metric_dictionary.csv
FOUND: data/processed/nhpd_san_diego_properties.csv
FOUND: data/processed/nhpd_san_diego_subsidies.csv
FOUND: data/processed/nhpd_san_diego_by_jurisdiction.csv
FOUND: data/processed/housing_need_affordability_validation_report.csv
FOUND: data/processed/housing_need_affordability_external_validation.csv
FOUND: data/processed/nhpd_san_diego_subsidy_expiration_risk.csv

All expected output files were created.
